# 02a — Phase 1 Screening v2 (Optuna params + 80/20 split)

Versi revisi dari notebook 02 agar metodologi evaluasi **konsisten dengan notebook 04**:

| Aspek | Notebook 02 (lama) | Notebook 02a (ini) |
|-------|--------------------|--------------------|
| Hyperparameter | Default konservatif | **Optuna-tuned** |
| Evaluasi | 5-fold expanding CV | **80/20 temporal split** |
| Metrik | MAPE mean ± std | **MAPE, MASE, DA, hit_large** |

## Desain
- **Models (4):** RandomForest, ExtraTrees, XGBoost, LightGBM
- **Covariates (16):** None (baseline) + 15 single covariates
- **Windows (2):** 20, 120 (business days)
- **Horizons (3):** 1, 5, 20 (business days)
- **Total:** 4 × 16 × 2 × 3 = **384 experiments**

## Output
- `phase1a_screening_results.csv` — 384 rows
- `phase1a_screening_analysis.csv` — pass/fail per covariate

In [1]:
import pandas as pd
import numpy as np
import joblib
import gc
import glob
import traceback
from datetime import datetime
import warnings

from darts import TimeSeries
from darts.models import RandomForestModel, XGBModel, LightGBMModel
from darts.dataprocessing.transformers import Scaler, Diff
from darts.utils.missing_values import fill_missing_values
from sklearn.ensemble import ExtraTreesRegressor

try:
    from darts.models import SKLearnModel
except ImportError:
    from darts.models import RegressionModel as SKLearnModel

warnings.filterwarnings("ignore")

LEVEL_VARS  = ["M2", "USDIDR", "Coal", "Copper", "Nickel", "Silver", "Tin", "STI", "Gold", "WTI", "GDP"]
RATE_VARS   = ["BI_Rate", "CPI", "NPL_Ratio", "US_Treasury_10Y"]
WINDOWS     = [20, 120]
HORIZONS    = [1, 5, 20]
TRAIN_RATIO = 0.8

# Load data
joblib_files = sorted(glob.glob("saved_models/df_merged_*.joblib"), reverse=True)
if not joblib_files:
    raise FileNotFoundError("df_merged_*.joblib tidak ditemukan. Jalankan notebook 00 dulu.")
df_merged = joblib.load(joblib_files[0])
print(f"Loaded: {joblib_files[0]}")
print(f"Data  : {df_merged.shape} | {df_merged['date'].min().date()} to {df_merged['date'].max().date()}")

# Load Optuna hyperparameters (dari notebook 01)
TUNING = joblib.load("saved_models/optuna_tuning_results.joblib")
print("\nOptuna best params:")
for m, r in TUNING.items():
    print(f"  {m}: best_value={r['best_value']:.4f} | params={r['best_params']}")

Loaded: saved_models/df_merged_20260507_1049.joblib
Data  : (2443, 17) | 2015-01-02 to 2025-01-31

Optuna best params:
  RandomForest: best_value=1.2634 | params={'n_estimators': 800, 'max_depth': 3, 'max_features': 0.7, 'max_samples': 0.8, 'min_samples_split': 5, 'min_samples_leaf': 6}
  ExtraTrees: best_value=1.2625 | params={'n_estimators': 200, 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_split': 9, 'min_samples_leaf': 3}
  XGBoost: best_value=1.2589 | params={'n_estimators': 600, 'max_depth': 11, 'learning_rate': 0.029464057132418377, 'subsample': 0.8256085472109767, 'colsample_bytree': 0.39195610746628573, 'reg_alpha': 4.723006221405656, 'reg_lambda': 0.0034555486843382047, 'min_child_weight': 8}
  LightGBM: best_value=1.2611 | params={'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.011711509955524094, 'num_leaves': 83, 'subsample': 0.5852620618436457, 'colsample_bytree': 0.3455361150896956, 'reg_alpha': 3.4671276804481113, 'reg_lambda': 4.905556676028774, 'min_c

## 2. Konfigurasi Eksperimen

In [2]:
SINGLE_COVARIATES = {
    "None":            [],        # baseline
    # Macro (7)
    "BI_Rate":         ["BI_Rate"],
    "CPI":             ["CPI"],
    "M2":              ["M2"],
    "NPL_Ratio":       ["NPL_Ratio"],
    "USDIDR":          ["USDIDR"],
    "GDP":             ["GDP"],
    "US_Treasury_10Y": ["US_Treasury_10Y"],
    # Commodity (7)
    "Coal":            ["Coal"],
    "Copper":          ["Copper"],
    "Nickel":          ["Nickel"],
    "Silver":          ["Silver"],
    "Tin":             ["Tin"],
    "Gold":            ["Gold"],
    "WTI":             ["WTI"],
    # Regional (1)
    "STI":             ["STI"],
}

total_exp = len(SINGLE_COVARIATES) * 4 * len(WINDOWS) * len(HORIZONS)
print(f"Covariates : {len(SINGLE_COVARIATES)} configs (termasuk None baseline)")
print(f"Models     : 4 (RF, ET, XGB, LGB)")
print(f"Windows    : {WINDOWS}")
print(f"Horizons   : {HORIZONS}")
print(f"Total      : {total_exp} experiments")

Covariates : 16 configs (termasuk None baseline)
Models     : 4 (RF, ET, XGB, LGB)
Windows    : [20, 120]
Horizons   : [1, 5, 20]
Total      : 384 experiments


## 3. Helper Functions

Pipeline identik dengan notebook 04 — hanya source covariate yang berbeda.

In [3]:
def to_series(df, target_col, covariates=None):
    target = TimeSeries.from_dataframe(
        df, time_col="date", value_cols=target_col,
        fill_missing_dates=True, freq="B",
    )
    target = fill_missing_values(target)
    cov = None
    if covariates:
        cov = TimeSeries.from_dataframe(
            df, time_col="date", value_cols=covariates,
            fill_missing_dates=True, freq="B",
        )
        cov = fill_missing_values(cov)
    return target, cov


def build_model(model_name, window, horizon, has_covariates):
    """Gunakan Optuna best_params — sama persis dengan notebook 04."""
    best_params = TUNING[model_name]['best_params'].copy()
    common = {
        "lags": window,
        "lags_past_covariates": window if has_covariates else None,
        "output_chunk_length": horizon,
    }
    if model_name == "RandomForest":
        return RandomForestModel(**common, random_state=42, n_jobs=-1, **best_params)
    elif model_name == "ExtraTrees":
        return SKLearnModel(
            **common,
            model=ExtraTreesRegressor(random_state=42, n_jobs=-1, **best_params),
        )
    elif model_name == "XGBoost":
        return XGBModel(**common, random_state=42, n_jobs=-1, **best_params)
    elif model_name == "LightGBM":
        return LightGBMModel(**common, random_state=42, n_jobs=-1, verbose=-1, **best_params)
    else:
        raise ValueError(f"Unknown model: {model_name}")


def transform_target(target_ts, split_idx):
    train_ts  = target_ts[:split_idx]
    full_log  = target_ts.map(np.log)
    train_log = train_ts.map(np.log)
    diff = Diff(lags=1)
    train_log_diff = diff.fit_transform(train_log)
    full_log_diff  = diff.transform(full_log)
    scaler = Scaler()
    train_scaled = scaler.fit_transform(train_log_diff)
    full_scaled  = scaler.transform(full_log_diff)
    return train_scaled, full_scaled, scaler


def transform_covariates(cov_ts, split_idx):
    if cov_ts is None:
        return None, None
    train_cov = cov_ts[:split_idx]
    full_cov  = cov_ts
    cov_cols   = cov_ts.components.tolist()
    level_cols = [c for c in cov_cols if c in LEVEL_VARS]
    rate_cols  = [c for c in cov_cols if c in RATE_VARS]
    parts_train, parts_full = [], []
    if level_cols:
        d = Diff(lags=1)
        parts_train.append(d.fit_transform(train_cov[level_cols].map(np.log)))
        parts_full.append(d.transform(full_cov[level_cols].map(np.log)))
    if rate_cols:
        d = Diff(lags=1)
        parts_train.append(d.fit_transform(train_cov[rate_cols]))
        parts_full.append(d.transform(full_cov[rate_cols]))
    ct, cf = parts_train[0], parts_full[0]
    for pt, pf in zip(parts_train[1:], parts_full[1:]):
        ct = ct.stack(pt)
        cf = cf.stack(pf)
    cov_scaler = Scaler()
    cov_scaler.fit(ct)
    return cov_scaler.transform(cf), ct.end_time()


def inverse_and_metrics(forecast_list, full_ts, scaler, split_idx):
    full_log = full_ts.map(np.log)
    all_dates, all_prices = [], []
    for chunk_scaled in forecast_list:
        chunk_diff = scaler.inverse_transform(chunk_scaled)
        dates = chunk_diff.time_index
        vals  = chunk_diff.values().flatten()
        idx   = full_ts.get_index_at_point(dates[0])
        if idx == 0:
            continue
        anchor     = full_log[idx - 1].values()[0][0]
        log_prices = anchor + np.cumsum(vals)
        all_dates.extend(dates)
        all_prices.extend(np.exp(log_prices))

    pred_df   = pd.DataFrame({"date": pd.to_datetime(all_dates), "predicted": all_prices})
    actual_df = full_ts.to_dataframe().reset_index()
    actual_df.columns = ["date", "actual"]
    eval_df   = pd.merge(actual_df, pred_df, on="date", how="inner")
    y_true    = eval_df["actual"].values
    y_pred    = eval_df["predicted"].values

    mape   = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    mae    = np.mean(np.abs(y_true - y_pred))
    rmse   = np.sqrt(np.mean((y_true - y_pred) ** 2))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2     = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    a_dir  = np.diff(y_true)
    p_dir  = y_pred[1:] - y_true[:-1]
    da     = np.mean((a_dir > 0) == (p_dir > 0)) * 100 if len(a_dir) > 0 else np.nan

    # MASE — skala-bebas, bandingkan dengan naive forecast training
    y_train = full_ts[:split_idx].values().flatten()
    naive_mae = np.mean(np.abs(np.diff(y_train))) if len(y_train) > 1 else np.nan
    mase = mae / naive_mae if (not np.isnan(naive_mae) and naive_mae > 0) else np.nan

    # Hit rate pada pergerakan besar (> 1 std return)
    if len(y_true) > 2 and len(a_dir) > 0:
        returns_test = np.diff(y_true) / y_true[:-1]
        sigma        = np.std(returns_test)
        large_mask   = np.abs(returns_test) > sigma
        hit_large    = (np.mean((a_dir[large_mask] > 0) == (p_dir[large_mask] > 0)) * 100
                        if large_mask.sum() > 0 else np.nan)
    else:
        hit_large = np.nan

    return {
        "mape":      round(mape, 4),
        "mae":       round(mae, 4),
        "rmse":      round(rmse, 4),
        "r2":        round(r2, 4),
        "da":        round(da, 2),
        "mase":      round(mase, 4) if not np.isnan(mase) else np.nan,
        "hit_large": round(hit_large, 2) if not np.isnan(hit_large) else np.nan,
    }


def evaluate_model(model_name, cov_name, cov_vars, window, horizon):
    """80/20 temporal split — identik dengan notebook 04."""
    target_ts, cov_ts = to_series(df_merged, "IHSG", cov_vars if cov_vars else None)
    n         = len(target_ts)
    split_idx = int(n * TRAIN_RATIO)

    train_scaled, full_scaled, scaler = transform_target(target_ts, split_idx)
    full_cov_scaled, _                = transform_covariates(cov_ts, split_idx)

    model = build_model(model_name, window, horizon, has_covariates=bool(cov_vars))
    model.fit(train_scaled, past_covariates=full_cov_scaled)

    test_start    = target_ts[split_idx].start_time()
    forecast_list = model.historical_forecasts(
        series=full_scaled,
        past_covariates=full_cov_scaled,
        start=test_start,
        forecast_horizon=horizon,
        stride=horizon,
        retrain=False,
        last_points_only=False,
        verbose=False,
    )
    if isinstance(forecast_list, TimeSeries):
        forecast_list = [forecast_list]
    return inverse_and_metrics(forecast_list, target_ts, scaler, split_idx)


print("Helpers defined.")
print(f"Pipeline: Optuna params + {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)} temporal split")

Helpers defined.
Pipeline: Optuna params + 80/19 temporal split


## 4. Experiment Loop (384 experiments)

In [4]:
MODEL_NAMES = ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]
results  = []
failed   = []
exp_num  = 0
start_time = datetime.now()

print(f"Started : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total   : {total_exp} experiments")
print("=" * 80)

for model_name in MODEL_NAMES:
    for cov_name, cov_vars in SINGLE_COVARIATES.items():
        for window in WINDOWS:
            for horizon in HORIZONS:
                exp_num += 1
                tag = f"[{exp_num:>3}/{total_exp}] {model_name:13s} | {cov_name:16s} | W{window:>3}_H{horizon:>2}"
                print(f"{tag}", end=" ... ", flush=True)
                try:
                    m = evaluate_model(model_name, cov_name, cov_vars, window, horizon)
                    results.append({
                        "Model": model_name, "Covariates": cov_name,
                        "Window": window, "Horizon": horizon, **m,
                    })
                    print(f"MAPE={m['mape']:.4f}%  DA={m['da']:.1f}%  MASE={m['mase']:.4f}")
                except Exception as e:
                    failed.append({"tag": tag, "error": str(e)})
                    print(f"FAILED: {e}")
                    traceback.print_exc()
                finally:
                    gc.collect()

                # Auto-save setiap selesai 1 model (anti-disconnect)
                if exp_num % (len(SINGLE_COVARIATES) * len(WINDOWS) * len(HORIZONS)) == 0:
                    pd.DataFrame(results).to_csv("phase1a_screening_results.csv", index=False)
                    print(f"  → Checkpoint saved ({len(results)} rows)")

elapsed = datetime.now() - start_time
print("=" * 80)
print(f"Done in {elapsed} | {len(results)} OK, {len(failed)} failed")

df_results = pd.DataFrame(results)
df_results.to_csv("phase1a_screening_results.csv", index=False)
print(f"Saved: phase1a_screening_results.csv ({len(df_results)} rows)")

Started : 2026-05-08 23:11:28
Total   : 384 experiments
[  1/384] RandomForest  | None             | W 20_H 1 ... MAPE=0.5072%  DA=51.7%  MASE=0.9746
[  2/384] RandomForest  | None             | W 20_H 5 ... MAPE=0.8522%  DA=47.5%  MASE=1.6371
[  3/384] RandomForest  | None             | W 20_H20 ... MAPE=1.3920%  DA=49.9%  MASE=2.6870
[  4/384] RandomForest  | None             | W120_H 1 ... MAPE=0.5069%  DA=52.3%  MASE=0.9740
[  5/384] RandomForest  | None             | W120_H 5 ... MAPE=0.8526%  DA=47.9%  MASE=1.6378
[  6/384] RandomForest  | None             | W120_H20 ... MAPE=1.3892%  DA=49.3%  MASE=2.6809
[  7/384] RandomForest  | BI_Rate          | W 20_H 1 ... MAPE=0.5071%  DA=52.7%  MASE=0.9745
[  8/384] RandomForest  | BI_Rate          | W 20_H 5 ... MAPE=0.8522%  DA=47.7%  MASE=1.6373
[  9/384] RandomForest  | BI_Rate          | W 20_H20 ... MAPE=1.3917%  DA=49.9%  MASE=2.6865
[ 10/384] RandomForest  | BI_Rate          | W120_H 1 ... MAPE=0.5067%  DA=53.2%  MASE=0.9737
[ 11

## 5. Screening Analysis

Covariate dianggap **lolos** jika memenuhi salah satu:
- **MAPE improvement** ≥ 0.3% relatif terhadap baseline (None)
- **DA improvement** ≥ 1.0 percentage point

Threshold yang sama dengan notebook 02 enhanced screening.

In [5]:
MAPE_THRESHOLD = 0.3   # % relative improvement
DA_THRESHOLD   = 1.0   # percentage point improvement

def analyze_screening(df):
    rows = []
    for model_name in df["Model"].unique():
        for window in df["Window"].unique():
            for horizon in df["Horizon"].unique():
                mask   = ((df["Model"] == model_name) &
                          (df["Window"] == window) &
                          (df["Horizon"] == horizon))
                subset = df[mask]
                bl     = subset[subset["Covariates"] == "None"]
                if bl.empty or "mape" not in bl.columns:
                    continue
                bl_mape = bl["mape"].values[0]
                bl_da   = bl["da"].values[0]

                for _, row in subset.iterrows():
                    if row["Covariates"] == "None" or pd.isna(row.get("mape")):
                        continue
                    mape_impr = (bl_mape - row["mape"]) / bl_mape * 100
                    da_impr   = row["da"] - bl_da
                    abs_impr  = bl_mape - row["mape"]
                    consistency = abs_impr / row["mape"] if row["mape"] > 0 else 0.0

                    rows.append({
                        "Model":         model_name,
                        "Window":        window,
                        "Horizon":       horizon,
                        "Covariate":     row["Covariates"],
                        "MAPE":          row["mape"],
                        "Baseline_MAPE": bl_mape,
                        "MAPE_Impr_pct": round(mape_impr, 3),
                        "DA":            row["da"],
                        "Baseline_DA":   round(bl_da, 2),
                        "DA_Impr_pp":    round(da_impr, 2),
                        "Consistency":   round(consistency, 4),
                        "MASE":          row.get("mase", np.nan),
                        "hit_large":     row.get("hit_large", np.nan),
                        "Passed_MAPE":   mape_impr >= MAPE_THRESHOLD,
                        "Passed_DA":     da_impr >= DA_THRESHOLD,
                        "Passed_Any":    (mape_impr >= MAPE_THRESHOLD) or (da_impr >= DA_THRESHOLD),
                        "Passed_Both":   (mape_impr >= MAPE_THRESHOLD) and (da_impr >= DA_THRESHOLD),
                    })
    return pd.DataFrame(rows)


df_screening = analyze_screening(df_results)
df_screening.to_csv("phase1a_screening_analysis.csv", index=False)
print(f"Saved: phase1a_screening_analysis.csv ({len(df_screening)} rows)")

Saved: phase1a_screening_analysis.csv (360 rows)


In [6]:
max_combos = (df_results["Model"].nunique()
              * df_results["Window"].nunique()
              * df_results["Horizon"].nunique())

ranking = (
    df_screening.groupby("Covariate")
    .agg(
        pass_mape       = ("Passed_MAPE", "sum"),
        pass_da         = ("Passed_DA",   "sum"),
        pass_any        = ("Passed_Any",  "sum"),
        pass_both       = ("Passed_Both", "sum"),
        avg_mape_impr   = ("MAPE_Impr_pct", "mean"),
        avg_da_impr     = ("DA_Impr_pp",    "mean"),
        avg_mase        = ("MASE",          "mean"),
        avg_hit_large   = ("hit_large",     "mean"),
        avg_consistency = ("Consistency",   "mean"),
    )
    .assign(
        pass_rate = lambda x: (x["pass_any"] / max_combos * 100).round(1),
        avg_mape_impr   = lambda x: x["avg_mape_impr"].round(3),
        avg_da_impr     = lambda x: x["avg_da_impr"].round(2),
        avg_consistency = lambda x: x["avg_consistency"].round(4),
    )
    .sort_values("pass_any", ascending=False)
)

print(f"=== COVARIATE RANKING — v2 (Optuna + 80/20) ===")
print(f"Max combos: {max_combos}  |  Threshold: MAPE ≥{MAPE_THRESHOLD}% atau DA ≥+{DA_THRESHOLD}pp\n")
display(ranking[[
    "pass_mape", "pass_da", "pass_any", "pass_rate",
    "avg_mape_impr", "avg_da_impr", "avg_mase", "avg_hit_large"
]].rename(columns={
    "pass_mape":    "Pass MAPE",
    "pass_da":      "Pass DA",
    "pass_any":     "Pass Any",
    "pass_rate":    "Pass Rate %",
    "avg_mape_impr":  "Avg MAPE Impr%",
    "avg_da_impr":    "Avg DA Impr pp",
    "avg_mase":       "Avg MASE",
    "avg_hit_large":  "Avg Hit Large%",
}))

# Tier summary
print("\n=== TIER SUMMARY ===")
tiers = {
    f"Tier 1 — Kuat (pass_any >= {int(max_combos*0.5)})":
        ranking[ranking["pass_any"] >= max_combos * 0.5],
    f"Tier 2 — Moderat (pass_any {int(max_combos*0.25)}-{int(max_combos*0.5)-1})":
        ranking[(ranking["pass_any"] >= max_combos * 0.25) & (ranking["pass_any"] < max_combos * 0.5)],
    f"Tier 3 — Lemah (pass_any < {int(max_combos*0.25)})":
        ranking[ranking["pass_any"] < max_combos * 0.25],
}
for label, tier in tiers.items():
    if tier.empty: continue
    print(f"\n{label}:")
    for cov, row in tier.iterrows():
        print(f"  {cov:18s} Pass={int(row['pass_any'])}/{max_combos}  "
              f"MAPEImpr={row['avg_mape_impr']:+.3f}%  "
              f"DA={row['avg_da_impr']:+.2f}pp  "
              f"MASE={row['avg_mase']:.4f}")

=== COVARIATE RANKING — v2 (Optuna + 80/20) ===
Max combos: 24  |  Threshold: MAPE ≥0.3% atau DA ≥+1.0pp



,Pass MAPE,Pass DA,Pass Any,Pass Rate %,Avg MAPE Impr%,Avg DA Impr pp,Avg MASE,Avg Hit Large%
Covariate,,,,,,,,
Silver,13,9,16,66.7,0.637,0.91,1.752842,53.471667
WTI,9,7,13,54.2,0.190,0.73,1.761787,53.382083
Gold,11,5,11,45.8,0.402,0.15,1.757479,51.861667
STI,9,2,9,37.5,0.317,-0.23,1.758808,51.074167
USDIDR,3,6,8,33.3,-0.281,0.16,1.773700,51.647500
Coal,6,2,7,29.2,0.107,-0.05,1.763808,52.502500
Nickel,5,2,7,29.2,-0.382,-0.62,1.772837,51.833333
CPI,3,4,6,25.0,-0.019,0.29,1.766075,52.317083
NPL_Ratio,4,2,6,25.0,0.100,0.02,1.762837,52.439583



=== TIER SUMMARY ===

Tier 1 — Kuat (pass_any >= 12):
  Silver             Pass=16/24  MAPEImpr=+0.637%  DA=+0.91pp  MASE=1.7528
  WTI                Pass=13/24  MAPEImpr=+0.190%  DA=+0.73pp  MASE=1.7618

Tier 2 — Moderat (pass_any 6-11):
  Gold               Pass=11/24  MAPEImpr=+0.402%  DA=+0.15pp  MASE=1.7575
  STI                Pass=9/24  MAPEImpr=+0.317%  DA=-0.23pp  MASE=1.7588
  USDIDR             Pass=8/24  MAPEImpr=-0.281%  DA=+0.16pp  MASE=1.7737
  Coal               Pass=7/24  MAPEImpr=+0.107%  DA=-0.05pp  MASE=1.7638
  Nickel             Pass=7/24  MAPEImpr=-0.382%  DA=-0.62pp  MASE=1.7728
  CPI                Pass=6/24  MAPEImpr=-0.019%  DA=+0.29pp  MASE=1.7661
  NPL_Ratio          Pass=6/24  MAPEImpr=+0.100%  DA=+0.02pp  MASE=1.7628
  Tin                Pass=6/24  MAPEImpr=+0.105%  DA=-0.17pp  MASE=1.7644

Tier 3 — Lemah (pass_any < 6):
  Copper             Pass=5/24  MAPEImpr=-0.202%  DA=-0.01pp  MASE=1.7683
  M2                 Pass=3/24  MAPEImpr=+0.076%  DA=-0.04pp 

## 6. Phase 2 Groups dari Hasil v2

In [7]:
MACRO_VARS     = ['BI_Rate', 'CPI', 'M2', 'NPL_Ratio', 'USDIDR', 'GDP', 'US_Treasury_10Y']
COMMODITY_VARS = ['Coal', 'Copper', 'Nickel', 'Silver', 'Tin', 'Gold', 'WTI']
REGIONAL_VARS  = ['STI']

THRESHOLD_PASS = max(2, int(max_combos * 0.25))  # >= 25% dari total combos
passed = ranking[ranking["pass_any"] >= THRESHOLD_PASS].index.tolist()

print(f"Threshold : >= {THRESHOLD_PASS} dari {max_combos} combos ({THRESHOLD_PASS/max_combos*100:.0f}%)")
print(f"Passed    : {passed}\n")

sig_macro     = [v for v in MACRO_VARS     if v in passed]
sig_commodity = [v for v in COMMODITY_VARS if v in passed]
sig_regional  = [v for v in REGIONAL_VARS  if v in passed]
sig_all       = sig_macro + sig_commodity + sig_regional

GROUP_COVARIATES_V2 = {}
if sig_macro:     GROUP_COVARIATES_V2['Sig_Macro']     = sig_macro
if sig_commodity: GROUP_COVARIATES_V2['Sig_Commodity'] = sig_commodity
if sig_regional:  GROUP_COVARIATES_V2['Sig_Regional']  = sig_regional
if len(sig_all) > 1: GROUP_COVARIATES_V2['Sig_All']    = sig_all

print("Phase 2 Groups (v2):")
for k, v in GROUP_COVARIATES_V2.items():
    print(f"  {k} ({len(v)} vars): {v}")

import os
os.makedirs("saved_models", exist_ok=True)
joblib.dump(GROUP_COVARIATES_V2, "saved_models/phase2_groups_v2.joblib")
print(f"\nSaved: saved_models/phase2_groups_v2.joblib")
print(f"Total Phase 2 experiments: {len(GROUP_COVARIATES_V2)} groups × 4 models × {len(WINDOWS)} windows × {len(HORIZONS)} horizons"
      f" = {len(GROUP_COVARIATES_V2)*4*len(WINDOWS)*len(HORIZONS)}")

Threshold : >= 6 dari 24 combos (25%)
Passed    : ['Silver', 'WTI', 'Gold', 'STI', 'USDIDR', 'Coal', 'Nickel', 'CPI', 'NPL_Ratio', 'Tin']

Phase 2 Groups (v2):
  Sig_Macro (3 vars): ['CPI', 'NPL_Ratio', 'USDIDR']
  Sig_Commodity (6 vars): ['Coal', 'Nickel', 'Silver', 'Tin', 'Gold', 'WTI']
  Sig_Regional (1 vars): ['STI']
  Sig_All (10 vars): ['CPI', 'NPL_Ratio', 'USDIDR', 'Coal', 'Nickel', 'Silver', 'Tin', 'Gold', 'WTI', 'STI']

Saved: saved_models/phase2_groups_v2.joblib
Total Phase 2 experiments: 4 groups × 4 models × 2 windows × 3 horizons = 96


## 7. Perbandingan: Notebook 02 (lama) vs 02a (baru)

In [8]:
# Load hasil notebook 02 lama untuk perbandingan
try:
    df_old = pd.read_csv("phase1_screening_analysis.csv")
    has_old = True
    print("Notebook 02 (lama) loaded.")
except FileNotFoundError:
    has_old = False
    print("phase1_screening_analysis.csv tidak ditemukan — skip perbandingan.")

if has_old:
    # Pass counts lama
    old_col = "Passed" if "Passed" in df_old.columns else "Passed_Any"
    old_pass = (df_old[df_old[old_col]]
                .groupby("Covariate").size()
                .rename("pass_old"))

    # Pass counts baru
    new_pass = (df_screening[df_screening["Passed_Any"]]
                .groupby("Covariate").size()
                .rename("pass_new"))

    cmp = pd.concat([old_pass, new_pass], axis=1).fillna(0).astype(int)
    cmp["delta"] = cmp["pass_new"] - cmp["pass_old"]
    cmp = cmp.sort_values("pass_new", ascending=False)

    print("\n=== Perbandingan Pass Count: lama (02) vs baru (02a) ===")
    print(f"{'Covariate':<20} {'Old':>5} {'New':>5} {'Delta':>7}")
    print("-" * 38)
    for cov, row in cmp.iterrows():
        delta_str = f"{row['delta']:+d}"
        flag = "▲" if row['delta'] > 0 else ("▼" if row['delta'] < 0 else " ")
        print(f"{cov:<20} {int(row['pass_old']):>5} {int(row['pass_new']):>5}  {flag} {delta_str:>4}")

Notebook 02 (lama) loaded.

=== Perbandingan Pass Count: lama (02) vs baru (02a) ===
Covariate              Old   New   Delta
--------------------------------------
Silver                  14    16  ▲   +2
WTI                      9    13  ▲   +4
Gold                    10    11  ▲   +1
STI                     11     9  ▼   -2
USDIDR                   7     8  ▲   +1
Coal                     1     7  ▲   +6
Nickel                   2     7  ▲   +5
CPI                      6     6      +0
NPL_Ratio                8     6  ▼   -2
Tin                      3     6  ▲   +3
Copper                  10     5  ▼   -5
M2                       9     3  ▼   -6
BI_Rate                  5     2  ▼   -3
GDP                      6     2  ▼   -4
US_Treasury_10Y          0     2  ▲   +2


## 8. Top 5 per Skenario (Window × Horizon) vs Baseline

Top 5 konfigurasi terbaik (Model + Covariate) diurutkan MAPE terendah untuk setiap skenario. Baseline ditampilkan sebagai referensi perbandingan.

In [9]:
df_valid = df_results.dropna(subset=["mape"]).copy()

SHOW_COLS = ["Model", "Covariates", "mape", "da", "mase", "hit_large"]

scenarios = sorted(
    df_valid[["Window", "Horizon"]].drop_duplicates().itertuples(index=False),
    key=lambda x: (x.Horizon, x.Window),
)

for sc in scenarios:
    w, h = sc.Window, sc.Horizon
    subset = df_valid[(df_valid["Window"] == w) & (df_valid["Horizon"] == h)]

    # Baseline rows (semua model, covariate=None)
    bl = subset[subset["Covariates"] == "None"][SHOW_COLS].copy()
    bl.insert(0, "Rank", "BL")

    # Top 5 non-baseline, sorted by MAPE ascending
    top5 = (
        subset[subset["Covariates"] != "None"]
        .sort_values("mape")
        .head(5)[SHOW_COLS]
        .reset_index(drop=True)
    )
    top5.insert(0, "Rank", [f"#{i+1}" for i in range(len(top5))])

    # Hitung improvement top5 vs rata-rata baseline
    bl_mape_avg = bl["mape"].mean()
    top5["impr_vs_bl%"] = ((bl_mape_avg - top5["mape"]) / bl_mape_avg * 100).round(3)

    print(f"\n{'='*68}")
    print(f"  Skenario W{w}_H{h}")
    print(f"{'='*68}")

    print(f"\n  Baseline (avg {len(bl)} model): MAPE = {bl_mape_avg:.4f}%")
    print("  " + "-" * 64)
    bl_show = bl.sort_values("mape")[["Rank","Model","Covariates","mape","da","mase"]].copy()
    bl_show.index = range(len(bl_show))
    print("  " + bl_show.to_string(index=False).replace("\n", "\n  "))

    print(f"\n  Top 5 Terbaik:")
    print("  " + "-" * 64)
    top5_show = top5[["Rank","Model","Covariates","mape","da","mase","impr_vs_bl%"]].copy()
    top5_show.index = range(len(top5_show))
    print("  " + top5_show.to_string(index=False).replace("\n", "\n  "))


  Skenario W20_H1

  Baseline (avg 4 model): MAPE = 0.5054%
  ----------------------------------------------------------------
  Rank        Model Covariates   mape    da   mase
    BL      XGBoost       None 0.5044 54.75 0.9691
    BL     LightGBM       None 0.5046 54.18 0.9697
    BL   ExtraTrees       None 0.5053 53.61 0.9710
    BL RandomForest       None 0.5072 51.71 0.9746

  Top 5 Terbaik:
  ----------------------------------------------------------------
  Rank    Model Covariates   mape    da   mase  impr_vs_bl%
    #1  XGBoost     Silver 0.5022 55.70 0.9650        0.628
    #2  XGBoost        CPI 0.5027 55.89 0.9658        0.529
    #3 LightGBM     USDIDR 0.5028 53.42 0.9659        0.510
    #4  XGBoost       Coal 0.5029 53.61 0.9662        0.490
    #5  XGBoost  NPL_Ratio 0.5029 54.75 0.9662        0.490

  Skenario W120_H1

  Baseline (avg 4 model): MAPE = 0.5094%
  ----------------------------------------------------------------
  Rank        Model Covariates   mape    da

## 9. Top 5 per Algoritma

Top 5 konfigurasi terbaik (Covariate + Window + Horizon) untuk masing-masing algoritma, diurutkan berdasarkan MAPE terendah. Baseline tiap algoritma ditampilkan sebagai referensi.

In [10]:
MODEL_COLORS = {
    "RandomForest": "RF",
    "ExtraTrees":   "ET",
    "XGBoost":      "XGB",
    "LightGBM":     "LGB",
}

for model_name in ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]:
    subset = df_valid[df_valid["Model"] == model_name]

    # Baseline semua skenario untuk model ini
    bl = subset[subset["Covariates"] == "None"].copy()
    bl_mape_avg = bl["mape"].mean()
    bl_mape_min = bl["mape"].min()
    bl_mape_max = bl["mape"].max()

    # Top 5 non-baseline, sorted by MAPE ascending
    top5 = (
        subset[subset["Covariates"] != "None"]
        .sort_values("mape")
        .head(5)[["Covariates", "Window", "Horizon", "mape", "da", "mase", "hit_large"]]
        .reset_index(drop=True)
    )
    top5.insert(0, "Rank", [f"#{i+1}" for i in range(len(top5))])
    top5["impr_vs_bl%"] = ((bl_mape_avg - top5["mape"]) / bl_mape_avg * 100).round(3)

    tag = MODEL_COLORS[model_name]
    print(f"\n{'='*70}")
    print(f"  [{tag}] {model_name}")
    print(f"{'='*70}")
    print(f"  Baseline MAPE — avg semua skenario : {bl_mape_avg:.4f}%")
    print(f"  Baseline MAPE — range              : {bl_mape_min:.4f}% – {bl_mape_max:.4f}%")
    print(f"\n  Top 5 Terbaik ({model_name}):")
    print("  " + "-" * 66)
    show = top5[["Rank","Covariates","Window","Horizon","mape","da","mase","impr_vs_bl%"]].copy()
    show.index = range(len(show))
    print("  " + show.to_string(index=False).replace("\n", "\n  "))

print("\n\n=== RINGKASAN: Best config per algoritma (MAPE terkecil) ===")
print(f"{'Model':<14} {'Covariate':<18} {'W':>4} {'H':>4} {'MAPE':>8} {'DA':>6} {'Impr%':>8}")
print("-" * 62)
for model_name in ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]:
    subset = df_valid[df_valid["Model"] == model_name]
    bl_avg  = subset[subset["Covariates"] == "None"]["mape"].mean()
    best    = subset[subset["Covariates"] != "None"].sort_values("mape").iloc[0]
    impr    = (bl_avg - best["mape"]) / bl_avg * 100
    print(f"{model_name:<14} {best['Covariates']:<18} {int(best['Window']):>4} "
          f"{int(best['Horizon']):>4} {best['mape']:>8.4f} {best['da']:>6.1f} {impr:>+8.3f}%")


  [RF] RandomForest
  Baseline MAPE — avg semua skenario : 0.9167%
  Baseline MAPE — range              : 0.5069% – 1.3920%

  Top 5 Terbaik (RandomForest):
  ------------------------------------------------------------------
  Rank Covariates  Window  Horizon   mape    da   mase  impr_vs_bl%
    #1     Silver      20        1 0.5047 53.61 0.9697       44.943
    #2        WTI     120        1 0.5051 53.23 0.9705       44.899
    #3        WTI      20        1 0.5053 53.61 0.9709       44.877
    #4     Silver     120        1 0.5055 54.18 0.9713       44.856
    #5     Copper     120        1 0.5058 51.71 0.9718       44.823

  [ET] ExtraTrees
  Baseline MAPE — avg semua skenario : 0.9144%
  Baseline MAPE — range              : 0.5053% – 1.3850%

  Top 5 Terbaik (ExtraTrees):
  ------------------------------------------------------------------
  Rank Covariates  Window  Horizon   mape    da   mase  impr_vs_bl%
    #1     Silver     120        1 0.5045 53.99 0.9693       44.828
    #2

## 10. Top 5 Overall — Best Configuration dari Seluruh Eksperimen

In [14]:
# Baseline per kombinasi Model × Window × Horizon
bl_lookup = (
    df_valid[df_valid["Covariates"] == "None"]
    .set_index(["Model", "Window", "Horizon"])[["mape", "da"]]
)

bl_global = df_valid[df_valid["Covariates"] == "None"]["mape"].mean()

# Top 5 overall
top5_all = (
    df_valid[df_valid["Covariates"] != "None"]
    .sort_values("mape")
    .head(5)[["Model", "Covariates", "Window", "Horizon", "mape", "da", "mase", "hit_large"]]
    .reset_index(drop=True)
)
top5_all.index += 1

# Tambah baseline MAPE dan DA per baris
top5_all["baseline_mape"] = top5_all.apply(
    lambda r: bl_lookup.loc[(r["Model"], r["Window"], r["Horizon"]), "mape"]
    if (r["Model"], r["Window"], r["Horizon"]) in bl_lookup.index else float("nan"), axis=1
)
top5_all["baseline_da"] = top5_all.apply(
    lambda r: bl_lookup.loc[(r["Model"], r["Window"], r["Horizon"]), "da"]
    if (r["Model"], r["Window"], r["Horizon"]) in bl_lookup.index else float("nan"), axis=1
)
top5_all["impr_mape%"] = (
    (top5_all["baseline_mape"] - top5_all["mape"]) / top5_all["baseline_mape"] * 100
).round(3)
top5_all["impr_da_pp"] = (top5_all["da"] - top5_all["baseline_da"]).round(2)

print(f"Baseline MAPE global: {bl_global:.4f}%")
print(f"\n{'='*72}")
print(f"  TOP 5 OVERALL — dari {len(df_valid[df_valid['Covariates'] != 'None'])} eksperimen non-baseline")
print(f"{'='*72}")
display(
    top5_all[[
        "Model", "Covariates", "Window", "Horizon",
        "baseline_mape", "mape", "impr_mape%",
        "baseline_da",   "da",   "impr_da_pp",
        "mase", "hit_large"
    ]]
    .rename(columns={
        "baseline_mape": "Baseline MAPE (%)",
        "mape":          "MAPE (%)",
        "impr_mape%":    "Impr MAPE (%)",
        "baseline_da":   "Baseline DA (%)",
        "da":            "DA (%)",
        "impr_da_pp":    "Impr DA (pp)",
        "mase":          "MASE",
        "hit_large":     "Hit Large (%)",
    })
    .style
    .format({
        "Baseline MAPE (%)": "{:.4f}",
        "MAPE (%)":          "{:.4f}",
        "Impr MAPE (%)":     "{:+.3f}",
        "Baseline DA (%)":   "{:.1f}",
        "DA (%)":            "{:.1f}",
        "Impr DA (pp)":      "{:+.2f}",
        "MASE":              "{:.4f}",
        "Hit Large (%)":     "{:.2f}",
    })
    .bar(subset=["MAPE (%)"],      color="#d65f5f", vmin=top5_all["mape"].min() * 0.998)
    .bar(subset=["Impr MAPE (%)"], color="#5fba7d", vmin=0)
    .bar(subset=["Impr DA (pp)"],  color="#42a5f5", vmin=0)
    .set_caption("Top 5 konfigurasi terbaik dari seluruh 384 eksperimen (diurutkan MAPE terkecil)")
)

Baseline MAPE global: 0.9169%

  TOP 5 OVERALL — dari 360 eksperimen non-baseline


,Model,Covariates,Window,Horizon,Baseline MAPE (%),MAPE (%),Impr MAPE (%),Baseline DA (%),DA (%),Impr DA (pp),MASE,Hit Large (%)
1,XGBoost,Silver,20,1,0.5044,0.5022,+0.436,54.8,55.7,+0.95,0.9650,54.74
2,XGBoost,CPI,20,1,0.5044,0.5027,+0.337,54.8,55.9,+1.14,0.9658,54.74
3,LightGBM,USDIDR,20,1,0.5046,0.5028,+0.357,54.2,53.4,-0.76,0.9659,52.55
4,XGBoost,NPL_Ratio,20,1,0.5044,0.5029,+0.297,54.8,54.8,+0.00,0.9662,56.20
5,XGBoost,Coal,20,1,0.5044,0.5029,+0.297,54.8,53.6,-1.14,0.9662,49.64


In [35]:
# ── TOP 3 OVERALL ────────────────────────────────────────────────────────────
top3_all = (
    df_valid[df_valid["Covariates"] != "None"]
    .sort_values("mape")
    .head(3)[["Model","Covariates","Window","Horizon","mape","da","mase","hit_large"]]
    .reset_index(drop=True)
)
top3_all.index += 1

top3_all["baseline_mape"] = top3_all.apply(
    lambda r: bl_lookup.loc[(r["Model"], r["Window"], r["Horizon"]), "mape"]
    if (r["Model"], r["Window"], r["Horizon"]) in bl_lookup.index else float("nan"), axis=1
)
top3_all["baseline_da"] = top3_all.apply(
    lambda r: bl_lookup.loc[(r["Model"], r["Window"], r["Horizon"]), "da"]
    if (r["Model"], r["Window"], r["Horizon"]) in bl_lookup.index else float("nan"), axis=1
)
top3_all["impr_mape%"] = ((top3_all["baseline_mape"] - top3_all["mape"]) / top3_all["baseline_mape"] * 100).round(3)
top3_all["impr_da_pp"] = (top3_all["da"] - top3_all["baseline_da"]).round(2)

print(f"{'='*65}")
print(f"  TOP 3 OVERALL — Single Covariate Experiments")
print(f"{'='*65}")
display(
    top3_all[["Model","Covariates","Window","Horizon",
              "baseline_mape","mape","impr_mape%",
              "baseline_da","da","impr_da_pp","mase"]]
    .rename(columns={
        "baseline_mape":"Baseline MAPE (%)","mape":"MAPE (%)",
        "impr_mape%":"Impr MAPE (%)","baseline_da":"Baseline DA (%)",
        "da":"DA (%)","impr_da_pp":"Impr DA (pp)","mase":"MASE",
    })
    .style
    .format({
        "Baseline MAPE (%)":"{:.4f}","MAPE (%)":"{:.4f}",
        "Impr MAPE (%)":"{:+.3f}","Baseline DA (%)":"{:.1f}",
        "DA (%)":"{:.1f}","Impr DA (pp)":"{:+.2f}","MASE":"{:.4f}",
    })
    .bar(subset=["Impr MAPE (%)"], color="#a5d6a7", vmin=0)
    .set_caption("Top 3 konfigurasi terbaik — divisualisasikan di notebook 04_visualization.ipynb")
)

# Simpan config top 3 untuk notebook visualisasi
import joblib as _jl, os as _os
_os.makedirs("saved_models", exist_ok=True)
top3_config = top3_all[["Model","Covariates","Window","Horizon"]].to_dict(orient="records")
_jl.dump(top3_config, "saved_models/top3_config.joblib")
print(f"\nSaved: saved_models/top3_config.joblib")
for i, c in enumerate(top3_config, 1):
    print(f"  #{i}: {c['Model']} | {c['Covariates']} | W{c['Window']}_H{c['Horizon']}")

  TOP 3 OVERALL — Single Covariate Experiments


,Model,Covariates,Window,Horizon,Baseline MAPE (%),MAPE (%),Impr MAPE (%),Baseline DA (%),DA (%),Impr DA (pp),MASE
1,XGBoost,Silver,20,1,0.5044,0.5022,+0.436,54.8,55.7,+0.95,0.9650
2,XGBoost,CPI,20,1,0.5044,0.5027,+0.337,54.8,55.9,+1.14,0.9658
3,LightGBM,USDIDR,20,1,0.5046,0.5028,+0.357,54.2,53.4,-0.76,0.9659



Saved: saved_models/top3_config.joblib
  #1: XGBoost | Silver | W20_H1
  #2: XGBoost | CPI | W20_H1
  #3: LightGBM | USDIDR | W20_H1


---
## 11. Tabel Ringkasan Performa

Tabel 1–3: ringkasan rata-rata per model dan skenario.  
Tabel 4–7: detail per covariate untuk masing-masing algoritma.

In [17]:
import warnings
warnings.filterwarnings("ignore")

# ── Helper: format style untuk tabel ringkasan ─────────────────────────────
def style_summary(df, mape_col="avg_mape", da_col="avg_da"):
    fmt = {c: "{:.4f}" for c in df.columns if "mape" in c.lower()}
    fmt.update({c: "{:.2f}" for c in df.columns if "da" in c.lower() or "impr" in c.lower()})
    return df.style.format(fmt)


# ══════════════════════════════════════════════════════════════════════════════
# Tabel 1 — Rata-rata MAPE & DA per Model (avg semua covariate + skenario)
# ══════════════════════════════════════════════════════════════════════════════
t1 = (
    df_valid
    .groupby("Model")
    .agg(
        avg_mape = ("mape", "mean"),
        avg_da   = ("da",   "mean"),
        n_exp    = ("mape", "count"),
    )
    .round({"avg_mape": 4, "avg_da": 2})
    .loc[["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]]
)

print("=" * 55)
print("  Tabel 1 — Rata-rata MAPE & DA per Model")
print("  (avg semua covariates × windows × horizons)")
print("=" * 55)
display(
    t1.rename(columns={"avg_mape": "Avg MAPE (%)", "avg_da": "Avg DA (%)", "n_exp": "N Eksperimen"})
    .style.format({"Avg MAPE (%)": "{:.4f}", "Avg DA (%)": "{:.2f}"})
    .bar(subset=["Avg MAPE (%)"], color="#ef9a9a", vmin=t1["avg_mape"].min() * 0.99)
    .bar(subset=["Avg DA (%)"],   color="#a5d6a7", vmin=50)
    .set_caption("Tabel 1: Rata-rata performa tiap algoritma (semua konfigurasi)")
)

# ══════════════════════════════════════════════════════════════════════════════
# Tabel 2 — Rata-rata MAPE & DA per Skenario × Model (pivot)
# ══════════════════════════════════════════════════════════════════════════════
df_valid["Skenario"] = "W" + df_valid["Window"].astype(str) + "_H" + df_valid["Horizon"].astype(str)

t2_raw = (
    df_valid
    .groupby(["Skenario", "Model"])
    .agg(avg_mape=("mape","mean"), avg_da=("da","mean"))
    .round({"avg_mape": 4, "avg_da": 2})
    .reset_index()
)

# Pivot MAPE
t2_mape = t2_raw.pivot(index="Skenario", columns="Model", values="avg_mape")
t2_mape = t2_mape[["RandomForest","ExtraTrees","XGBoost","LightGBM"]]
t2_mape.columns = [f"{c}\nMAPE%" for c in t2_mape.columns]

# Pivot DA
t2_da = t2_raw.pivot(index="Skenario", columns="Model", values="avg_da")
t2_da = t2_da[["RandomForest","ExtraTrees","XGBoost","LightGBM"]]
t2_da.columns = [f"{c}\nDA%" for c in t2_da.columns]

t2 = pd.concat([t2_mape, t2_da], axis=1)
t2 = t2.loc[["W20_H1","W120_H1","W20_H5","W120_H5","W20_H20","W120_H20"]]

print("\n" + "=" * 55)
print("  Tabel 2 — Avg MAPE & DA per Skenario × Model")
print("  (avg semua covariates pada skenario tersebut)")
print("=" * 55)
display(
    t2.style
    .format({c: "{:.4f}" for c in t2.columns if "MAPE" in c})
    .format({c: "{:.2f}" for c in t2.columns if "DA" in c})
    .set_caption("Tabel 2: Rata-rata performa per skenario × algoritma")
)

# ══════════════════════════════════════════════════════════════════════════════
# Tabel 3 — Rata-rata MAPE & DA keseluruhan per Model (1 angka per model)
# ══════════════════════════════════════════════════════════════════════════════
t3 = (
    df_valid
    .groupby("Model")
    .agg(
        avg_mape     = ("mape", "mean"),
        std_mape     = ("mape", "std"),
        best_mape    = ("mape", "min"),
        avg_da       = ("da",   "mean"),
        std_da       = ("da",   "std"),
        best_da      = ("da",   "max"),
    )
    .round(4)
    .loc[["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]]
)

print("\n" + "=" * 65)
print("  Tabel 3 — Summary Keseluruhan per Algoritma (semua dirata-ratakan)")
print("=" * 65)
display(
    t3.rename(columns={
        "avg_mape":  "Avg MAPE (%)",
        "std_mape":  "Std MAPE",
        "best_mape": "Best MAPE (%)",
        "avg_da":    "Avg DA (%)",
        "std_da":    "Std DA",
        "best_da":   "Best DA (%)",
    })
    .style
    .format({
        "Avg MAPE (%)": "{:.4f}", "Std MAPE":    "{:.4f}", "Best MAPE (%)": "{:.4f}",
        "Avg DA (%)":   "{:.2f}", "Std DA":       "{:.2f}", "Best DA (%)":   "{:.2f}",
    })
    .highlight_min(subset=["Avg MAPE (%)"],  color="#c8e6c9")
    .highlight_min(subset=["Best MAPE (%)"], color="#a5d6a7")
    .highlight_max(subset=["Best DA (%)"],   color="#bbdefb")
    .set_caption("Tabel 3: Ringkasan performa keseluruhan tiap algoritma")
)

  Tabel 1 — Rata-rata MAPE & DA per Model
  (avg semua covariates × windows × horizons)


,Avg MAPE (%),Avg DA (%),N Eksperimen
Model,,,
RandomForest,0.9202,49.79,96
ExtraTrees,0.9150,50.00,96
XGBoost,0.9181,49.31,96
LightGBM,0.9173,49.53,96



  Tabel 2 — Avg MAPE & DA per Skenario × Model
  (avg semua covariates pada skenario tersebut)


,RandomForest MAPE%,ExtraTrees MAPE%,XGBoost MAPE%,LightGBM MAPE%,RandomForest DA%,ExtraTrees DA%,XGBoost DA%,LightGBM DA%
Skenario,,,,,,,,
W20_H1,0.507400,0.505400,0.504600,0.505200,51.94,52.93,53.36,53.03
W120_H1,0.507300,0.505600,0.513400,0.510100,51.78,52.87,49.86,51.09
W20_H5,0.854200,0.852700,0.851500,0.849800,47.73,48.09,47.91,47.86
W120_H5,0.854400,0.852700,0.856900,0.851800,47.95,48.03,47.83,48.00
W20_H20,1.401100,1.388300,1.388000,1.394400,49.83,48.82,48.77,49.29
W120_H20,1.396400,1.385500,1.394200,1.392600,49.53,49.24,48.11,47.93



  Tabel 3 — Summary Keseluruhan per Algoritma (semua dirata-ratakan)


,Avg MAPE (%),Std MAPE,Best MAPE (%),Avg DA (%),Std DA,Best DA (%)
Model,,,,,,
RandomForest,0.9202,0.3692,0.5047,49.79,1.91,54.18
ExtraTrees,0.9150,0.3644,0.5045,50.00,2.25,56.27
XGBoost,0.9181,0.3651,0.5022,49.31,2.26,55.89
LightGBM,0.9173,0.3667,0.5028,49.53,2.24,55.51


In [18]:
def tabel_per_model(model_name):
    """
    Tabel MAPE & DA semua covariate untuk satu model.
    Nilai = rata-rata across semua window × horizon.
    Baseline (None) ditampilkan di baris pertama sebagai referensi.
    """
    sub = df_valid[df_valid["Model"] == model_name].copy()

    # Avg per covariate (across all W × H)
    agg = (
        sub.groupby("Covariates")
        .agg(avg_mape=("mape","mean"), avg_da=("da","mean"))
        .round({"avg_mape": 4, "avg_da": 2})
        .reset_index()
    )

    # Baseline row
    bl_row  = agg[agg["Covariates"] == "None"].iloc[0]
    bl_mape = bl_row["avg_mape"]
    bl_da   = bl_row["avg_da"]

    # Non-baseline: hitung improvement
    non_bl = agg[agg["Covariates"] != "None"].copy()
    non_bl["impr_mape%"] = ((bl_mape - non_bl["avg_mape"]) / bl_mape * 100).round(3)
    non_bl["impr_da_pp"] = (non_bl["avg_da"] - bl_da).round(2)
    non_bl = non_bl.sort_values("avg_mape").reset_index(drop=True)
    non_bl.index += 1  # rank mulai dari 1

    # Baseline row (tanpa improvement — itu referensi)
    bl_display = pd.DataFrame([{
        "Covariates":  "None (Baseline)",
        "avg_mape":    bl_mape,
        "avg_da":      bl_da,
        "impr_mape%":  0.0,
        "impr_da_pp":  0.0,
    }], index=["BL"])

    tbl = pd.concat([bl_display, non_bl])

    tag = {"RandomForest":"RF","ExtraTrees":"ET","XGBoost":"XGB","LightGBM":"LGB"}[model_name]
    print(f"\n{'='*70}")
    print(f"  Tabel [{tag}] {model_name}")
    print(f"  Avg MAPE & DA per Covariate (rata-rata semua W × H)")
    print(f"  Baseline MAPE = {bl_mape:.4f}%  |  Baseline DA = {bl_da:.2f}%")
    print(f"{'='*70}")
    display(
        tbl.rename(columns={
            "Covariates":  "Covariate",
            "avg_mape":    "Avg MAPE (%)",
            "avg_da":      "Avg DA (%)",
            "impr_mape%":  "Impr MAPE (%)",
            "impr_da_pp":  "Impr DA (pp)",
        })
        .style
        .format({
            "Avg MAPE (%)":  "{:.4f}",
            "Avg DA (%)":    "{:.2f}",
            "Impr MAPE (%)": "{:+.3f}",
            "Impr DA (pp)":  "{:+.2f}",
        })
        .apply(lambda col: [
            "background-color: #e8f5e9" if v > 0 else
            "background-color: #ffebee" if v < 0 else ""
            for v in col
        ], subset=["Impr MAPE (%)"])
        .apply(lambda col: [
            "background-color: #e3f2fd" if v > 0 else
            "background-color: #fff3e0" if v < 0 else ""
            for v in col
        ], subset=["Impr DA (pp)"])
        .set_caption(
            f"Tabel {tag}: {model_name} — MAPE & DA tiap covariate vs baseline "
            f"(hijau=impr MAPE, biru=impr DA)"
        )
    )

# Tabel 4–7: satu per algoritma
for model in ["RandomForest", "XGBoost", "ExtraTrees", "LightGBM"]:
    tabel_per_model(model)


  Tabel [RF] RandomForest
  Avg MAPE & DA per Covariate (rata-rata semua W × H)
  Baseline MAPE = 0.9167%  |  Baseline DA = 49.77%


,Covariate,Avg MAPE (%),Avg DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),0.9167,49.77,+0.000,+0.00
1,M2,0.9142,49.80,+0.273,+0.03
2,NPL_Ratio,0.9148,49.58,+0.207,-0.19
3,WTI,0.9148,50.22,+0.207,+0.45
4,Silver,0.9157,50.38,+0.109,+0.61
5,Copper,0.9163,49.71,+0.044,-0.06
6,GDP,0.9164,50.15,+0.033,+0.38
7,BI_Rate,0.9166,50.18,+0.011,+0.41
8,Coal,0.9170,49.80,-0.033,+0.03
9,CPI,0.9175,50.00,-0.087,+0.23



  Tabel [XGB] XGBoost
  Avg MAPE & DA per Covariate (rata-rata semua W × H)
  Baseline MAPE = 0.9194%  |  Baseline DA = 49.23%


,Covariate,Avg MAPE (%),Avg DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),0.9194,49.23,+0.000,+0.00
1,Silver,0.9070,50.34,+1.349,+1.11
2,STI,0.9071,48.94,+1.338,-0.29
3,Gold,0.9093,50.25,+1.099,+1.02
4,Tin,0.9152,49.16,+0.457,-0.07
5,WTI,0.9159,50.08,+0.381,+0.85
6,NPL_Ratio,0.9166,49.67,+0.305,+0.44
7,Coal,0.9167,49.46,+0.294,+0.23
8,BI_Rate,0.9172,49.07,+0.239,-0.16
9,GDP,0.9172,48.85,+0.239,-0.38



  Tabel [ET] ExtraTrees
  Avg MAPE & DA per Covariate (rata-rata semua W × H)
  Baseline MAPE = 0.9144%  |  Baseline DA = 49.99%


,Covariate,Avg MAPE (%),Avg DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),0.9144,49.99,+0.000,+0.00
1,NPL_Ratio,0.9137,49.61,+0.077,-0.38
2,Silver,0.9140,50.85,+0.044,+0.86
3,STI,0.9142,49.55,+0.022,-0.44
4,BI_Rate,0.9143,50.63,+0.011,+0.64
5,Coal,0.9143,49.84,+0.011,-0.15
6,Copper,0.9143,50.38,+0.011,+0.39
7,WTI,0.9143,50.25,+0.011,+0.26
8,M2,0.9144,49.80,+0.000,-0.19
9,USDIDR,0.9144,50.06,+0.000,+0.07



  Tabel [LGB] LightGBM
  Avg MAPE & DA per Covariate (rata-rata semua W × H)
  Baseline MAPE = 0.9172%  |  Baseline DA = 49.52%


,Covariate,Avg MAPE (%),Avg DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),0.9172,49.52,+0.000,+0.00
1,Silver,0.9058,50.59,+1.243,+1.07
2,Gold,0.9075,49.08,+1.058,-0.44
3,STI,0.9098,48.97,+0.807,-0.55
4,WTI,0.9148,50.88,+0.262,+1.36
5,Tin,0.9159,49.36,+0.142,-0.16
6,Coal,0.9164,49.23,+0.087,-0.29
7,BI_Rate,0.9172,49.52,+0.000,+0.00
8,GDP,0.9178,49.71,-0.065,+0.19
9,M2,0.9178,49.71,-0.065,+0.19


## 12. Tabel per Model × Skenario (Tabel 4–7 dipisah tiap Skenario)

Setiap tabel menampilkan MAPE & DA semua covariate untuk kombinasi spesifik **Model × Window × Horizon**. Total: 4 model × 6 skenario = 24 tabel.

In [19]:
MODEL_TAG = {
    "RandomForest": "RF",
    "ExtraTrees":   "ET",
    "XGBoost":      "XGB",
    "LightGBM":     "LGB",
}
SCENARIO_ORDER = ["W20_H1", "W120_H1", "W20_H5", "W120_H5", "W20_H20", "W120_H20"]

def tabel_model_skenario(model_name, window, horizon):
    """
    Tabel MAPE & DA semua covariate untuk satu model pada satu skenario.
    Baseline (None) di baris pertama, non-baseline diurutkan MAPE terkecil.
    """
    sub = df_valid[
        (df_valid["Model"]   == model_name) &
        (df_valid["Window"]  == window) &
        (df_valid["Horizon"] == horizon)
    ][["Covariates", "mape", "da"]].copy()

    if sub.empty:
        print(f"  [SKIP] {model_name} W{window}_H{horizon} — tidak ada data")
        return

    bl      = sub[sub["Covariates"] == "None"].iloc[0]
    bl_mape = bl["mape"]
    bl_da   = bl["da"]

    non_bl = sub[sub["Covariates"] != "None"].copy()
    non_bl["impr_mape%"] = ((bl_mape - non_bl["mape"]) / bl_mape * 100).round(3)
    non_bl["impr_da_pp"] = (non_bl["da"] - bl_da).round(2)
    non_bl = non_bl.sort_values("mape").reset_index(drop=True)
    non_bl.index += 1

    bl_row = pd.DataFrame([{
        "Covariates":  "None (Baseline)",
        "mape":        bl_mape,
        "da":          bl_da,
        "impr_mape%":  0.0,
        "impr_da_pp":  0.0,
    }], index=["BL"])

    tbl = pd.concat([bl_row, non_bl])
    tag = MODEL_TAG[model_name]

    print(f"\n{'─'*62}")
    print(f"  [{tag}] {model_name}  |  W{window}_H{horizon}")
    print(f"  Baseline → MAPE: {bl_mape:.4f}%  DA: {bl_da:.2f}%")
    print(f"{'─'*62}")
    display(
        tbl.rename(columns={
            "Covariates":  "Covariate",
            "mape":        "MAPE (%)",
            "da":          "DA (%)",
            "impr_mape%":  "Impr MAPE (%)",
            "impr_da_pp":  "Impr DA (pp)",
        })
        .style
        .format({
            "MAPE (%)":      "{:.4f}",
            "DA (%)":        "{:.2f}",
            "Impr MAPE (%)": "{:+.3f}",
            "Impr DA (pp)":  "{:+.2f}",
        })
        .apply(lambda col: [
            "background-color: #e8f5e9" if v > 0 else
            "background-color: #ffebee" if v < 0 else
            "background-color: #f5f5f5"
            for v in col
        ], subset=["Impr MAPE (%)"])
        .apply(lambda col: [
            "background-color: #e3f2fd" if v > 0 else
            "background-color: #fff3e0" if v < 0 else
            "background-color: #f5f5f5"
            for v in col
        ], subset=["Impr DA (pp)"])
        .set_caption(f"{tag} | W{window}_H{horizon} — MAPE & DA tiap covariate vs baseline")
    )


# ── Loop: 4 model × 6 skenario ─────────────────────────────────────────────
for model_name in ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]:
    tag = MODEL_TAG[model_name]
    print(f"\n{'#'*62}")
    print(f"  [{tag}] {model_name}")
    print(f"{'#'*62}")
    for sc in SCENARIO_ORDER:
        w, h = int(sc.split("_H")[0][1:]), int(sc.split("_H")[1])
        tabel_model_skenario(model_name, w, h)


##############################################################
  [RF] RandomForest
##############################################################

──────────────────────────────────────────────────────────────
  [RF] RandomForest  |  W20_H1
  Baseline → MAPE: 0.5072%  DA: 51.71%
──────────────────────────────────────────────────────────────


,Covariate,MAPE (%),DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),0.5072,51.71,+0.000,+0.00
1,Silver,0.5047,53.61,+0.493,+1.90
2,WTI,0.5053,53.61,+0.375,+1.90
3,Copper,0.5060,51.33,+0.237,-0.38
4,USDIDR,0.5063,53.23,+0.177,+1.52
5,GDP,0.5070,53.42,+0.039,+1.71
6,BI_Rate,0.5071,52.66,+0.020,+0.95
7,CPI,0.5071,52.85,+0.020,+1.14
8,M2,0.5071,52.28,+0.020,+0.57
9,NPL_Ratio,0.5071,52.66,+0.020,+0.95



──────────────────────────────────────────────────────────────
  [RF] RandomForest  |  W120_H1
  Baseline → MAPE: 0.5069%  DA: 52.28%
──────────────────────────────────────────────────────────────


,Covariate,MAPE (%),DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),0.5069,52.28,+0.000,+0.00
1,WTI,0.5051,53.23,+0.355,+0.95
2,Silver,0.5055,54.18,+0.276,+1.90
3,Copper,0.5058,51.71,+0.217,-0.57
4,USDIDR,0.5061,53.23,+0.158,+0.95
5,BI_Rate,0.5067,53.23,+0.039,+0.95
6,M2,0.5069,52.28,+0.000,+0.00
7,CPI,0.5070,52.28,-0.020,+0.00
8,Tin,0.5070,50.57,-0.020,-1.71
9,Coal,0.5072,51.14,-0.059,-1.14



──────────────────────────────────────────────────────────────
  [RF] RandomForest  |  W20_H5
  Baseline → MAPE: 0.8522%  DA: 47.52%
──────────────────────────────────────────────────────────────


...

,Covariate,MAPE (%),DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),0.8514,47.52,+0.000,+0.00
1,Gold,0.8370,45.61,+1.691,-1.91
2,STI,0.8401,47.90,+1.327,+0.38
3,Silver,0.8440,48.47,+0.869,+0.95
4,WTI,0.8474,50.57,+0.470,+3.05
5,USDIDR,0.8494,48.66,+0.235,+1.14
6,Nickel,0.8505,48.66,+0.106,+1.14
7,BI_Rate,0.8514,47.52,+0.000,+0.00
8,M2,0.8518,47.90,-0.047,+0.38
9,NPL_Ratio,0.8518,47.90,-0.047,+0.38



──────────────────────────────────────────────────────────────
  [LGB] LightGBM  |  W20_H20
  Baseline → MAPE: 1.3996%  DA: 49.33%
──────────────────────────────────────────────────────────────


,Covariate,MAPE (%),DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),1.3996,49.33,+0.000,+0.00
1,Silver,1.3645,49.71,+2.508,+0.38
2,STI,1.3819,48.17,+1.265,-1.16
3,Gold,1.3852,50.87,+1.029,+1.54
4,Tin,1.3867,48.75,+0.922,-0.58
5,Copper,1.3871,49.33,+0.893,+0.00
6,WTI,1.3892,48.55,+0.743,-0.78
7,Nickel,1.3933,49.52,+0.450,+0.19
8,CPI,1.3974,49.52,+0.157,+0.19
9,M2,1.3974,49.52,+0.157,+0.19



──────────────────────────────────────────────────────────────
  [LGB] LightGBM  |  W120_H20
  Baseline → MAPE: 1.3849%  DA: 47.21%
──────────────────────────────────────────────────────────────


,Covariate,MAPE (%),DA (%),Impr MAPE (%),Impr DA (pp)
BL,None (Baseline),1.3849,47.21,+0.000,+0.00
1,Gold,1.3712,49.52,+0.989,+2.31
2,Silver,1.3770,47.98,+0.570,+0.77
3,STI,1.3771,47.78,+0.563,+0.57
4,Coal,1.3800,48.17,+0.354,+0.96
5,BI_Rate,1.3849,47.21,+0.000,+0.00
6,WTI,1.3853,48.55,-0.029,+1.34
7,M2,1.3899,47.59,-0.361,+0.38
8,NPL_Ratio,1.3899,47.59,-0.361,+0.38
9,GDP,1.3899,47.59,-0.361,+0.38


## 13. Covariate Selection Helper

Tabel berikut merangkum semua informasi screening untuk membantu memilih covariate yang akan digunakan pada training ulang.

In [24]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

MACRO_VARS     = ["BI_Rate","CPI","M2","NPL_Ratio","USDIDR","GDP","US_Treasury_10Y"]
COMMODITY_VARS = ["Coal","Copper","Nickel","Silver","Tin","Gold","WTI"]
REGIONAL_VARS  = ["STI"]

def get_cat(cov):
    if cov in MACRO_VARS:     return "Macro"
    if cov in COMMODITY_VARS: return "Commodity"
    return "Regional"

# ── Agregasi dari df_screening (semua 24 kombinasi model×window×horizon) ──
max_combos = df_results["Model"].nunique() * df_results["Window"].nunique() * df_results["Horizon"].nunique()

sel = (
    df_screening
    .groupby("Covariate")
    .agg(
        pass_mape        = ("Passed_MAPE",    "sum"),
        pass_da          = ("Passed_DA",       "sum"),
        pass_any         = ("Passed_Any",      "sum"),
        pass_both        = ("Passed_Both",     "sum"),
        avg_mape_impr    = ("MAPE_Impr_pct",   "mean"),
        std_mape_impr    = ("MAPE_Impr_pct",   "std"),
        avg_da_impr      = ("DA_Impr_pp",      "mean"),
        avg_mase         = ("MASE",            "mean"),
        avg_hit_large    = ("hit_large",       "mean"),
        avg_consistency  = ("Consistency",     "mean"),
        # MAPE absolut rata-rata tiap horizon
        n_h1_pass        = ("Passed_Any",
                             lambda x: x[df_screening.loc[x.index,"Horizon"]==1].sum()),
        n_h5_pass        = ("Passed_Any",
                             lambda x: x[df_screening.loc[x.index,"Horizon"]==5].sum()),
        n_h20_pass       = ("Passed_Any",
                             lambda x: x[df_screening.loc[x.index,"Horizon"]==20].sum()),
    )
    .reset_index()
)

sel["Category"]      = sel["Covariate"].map(get_cat)
sel["pass_rate_%"]   = (sel["pass_any"]  / max_combos * 100).round(1)
sel["avg_mape_impr"] = sel["avg_mape_impr"].round(3)
sel["avg_da_impr"]   = sel["avg_da_impr"].round(2)
sel["avg_mase"]      = sel["avg_mase"].round(4)
sel["avg_hit_large"] = sel["avg_hit_large"].round(2)

# Tier berdasarkan: avg_mape_impr > 0 AND pass_any >= 25% threshold
sel["mape_positive"] = sel["avg_mape_impr"] > 0
sel["Tier"] = sel.apply(lambda r:
    "✅ Kuat"    if r["pass_any"] >= max_combos * 0.5  and r["mape_positive"] else
    "🟡 Moderat" if r["pass_any"] >= max_combos * 0.25 and r["mape_positive"] else
    "⚠️ Lemah"   if r["pass_any"] >= max_combos * 0.25 and not r["mape_positive"] else
    "❌ Tidak lolos",
    axis=1
)

# Sort: kuat dulu, lalu avg_mape_impr desc
tier_order = {"✅ Kuat": 0, "🟡 Moderat": 1, "⚠️ Lemah": 2, "❌ Tidak lolos": 3}
sel["tier_ord"] = sel["Tier"].map(tier_order)
sel = sel.sort_values(["tier_ord","avg_mape_impr"], ascending=[True, False]).drop(columns="tier_ord")

# ── Tampilkan tabel lengkap ──
DISPLAY_COLS = [
    "Covariate","Category","Tier",
    "pass_any","pass_rate_%",
    "pass_mape","pass_da","pass_both",
    "n_h1_pass","n_h5_pass","n_h20_pass",
    "avg_mape_impr","avg_da_impr",
    "avg_mase","avg_hit_large",
]

max_h_combos = df_results["Model"].nunique() * df_results["Window"].nunique()
print(f"Max combos total     : {max_combos}  (4 model × 2 window × 3 horizon)")
print(f"Max combos per horizon: {max_h_combos}  (4 model × 2 window)")
print(f"\nThreshold lolos: avg_mape_impr > 0 AND pass_any >= {int(max_combos*0.25)} ({int(100*0.25)}%)")

def color_tier(val):
    return {
        "✅ Kuat":       "background-color:#c8e6c9;font-weight:bold",
        "🟡 Moderat":    "background-color:#fff9c4",
        "⚠️ Lemah":      "background-color:#ffe0b2",
        "❌ Tidak lolos":"background-color:#ffcdd2",
    }.get(val, "")

def color_impr(val):
    try:
        v = float(val)
        if v > 0:   return "color:#2e7d32;font-weight:bold"
        if v < 0:   return "color:#c62828"
        return ""
    except: return ""

tbl = sel[DISPLAY_COLS].copy().reset_index(drop=True)
tbl.index += 1

display(
    tbl.rename(columns={
        "pass_any":      "Pass Any",
        "pass_rate_%":   "Pass Rate%",
        "pass_mape":     "Pass MAPE",
        "pass_da":       "Pass DA",
        "pass_both":     "Pass Both",
        "n_h1_pass":     "H1 Pass",
        "n_h5_pass":     "H5 Pass",
        "n_h20_pass":    "H20 Pass",
        "avg_mape_impr": "Avg MAPE Impr%",
        "avg_da_impr":   "Avg DA Impr(pp)",
        "avg_mase":      "Avg MASE",
        "avg_hit_large": "Avg Hit Large%",
    })
    .style
    .format({
        "Pass Rate%":      "{:.1f}",
        "Avg MAPE Impr%":  "{:+.3f}",
        "Avg DA Impr(pp)": "{:+.2f}",
        "Avg MASE":        "{:.4f}",
        "Avg Hit Large%":  "{:.2f}",
    })
    .map(color_tier, subset=["Tier"])
    .map(color_impr, subset=["Avg MAPE Impr%","Avg DA Impr(pp)"])
    .set_caption(
        f"Covariate Selection Helper — dari {max_combos} kombinasi (4 model × 2 window × 3 horizon)\n"
        "Tier: ✅Kuat = pass≥50% & MAPE+  🟡Moderat = pass≥25% & MAPE+  ⚠️Lemah = pass≥25% tapi MAPE-  ❌Tidak lolos"
    )
)

# ── Ringkasan rekomendasi ──
print("\n" + "="*60)
print("  REKOMENDASI COVARIATE UNTUK TRAINING")
print("="*60)
for tier in ["✅ Kuat","🟡 Moderat","⚠️ Lemah","❌ Tidak lolos"]:
    covs = sel[sel["Tier"]==tier]["Covariate"].tolist()
    cats = sel[sel["Tier"]==tier]["Category"].tolist()
    if covs:
        print(f"\n{tier}:")
        for c, cat in zip(covs, cats):
            row = sel[sel["Covariate"]==c].iloc[0]
            print(f"  {c:<18} [{cat:<9}]  Pass={int(row['pass_any'])}/{max_combos}  "
                  f"MAPEImpr={row['avg_mape_impr']:+.3f}%  "
                  f"DA={row['avg_da_impr']:+.2f}pp  "
                  f"MASE={row['avg_mase']:.4f}")


Max combos total     : 24  (4 model × 2 window × 3 horizon)
Max combos per horizon: 8  (4 model × 2 window)

Threshold lolos: avg_mape_impr > 0 AND pass_any >= 6 (25%)


,Covariate,Category,Tier,Pass Any,Pass Rate%,Pass MAPE,Pass DA,Pass Both,H1 Pass,H5 Pass,H20 Pass,Avg MAPE Impr%,Avg DA Impr(pp),Avg MASE,Avg Hit Large%
1,Silver,Commodity,✅ Kuat,16,66.7,13,9,6,7,5,4,+0.637,+0.91,1.7528,53.47
2,WTI,Commodity,✅ Kuat,13,54.2,9,7,3,6,3,4,+0.190,+0.73,1.7618,53.38
3,Gold,Commodity,🟡 Moderat,11,45.8,11,5,5,2,5,4,+0.402,+0.15,1.7575,51.86
4,STI,Regional,🟡 Moderat,9,37.5,9,2,2,2,3,4,+0.317,-0.23,1.7588,51.07
5,Coal,Commodity,🟡 Moderat,7,29.2,6,2,1,1,3,3,+0.107,-0.05,1.7638,52.50
6,Tin,Commodity,🟡 Moderat,6,25.0,6,2,2,2,1,3,+0.105,-0.17,1.7644,52.53
7,NPL_Ratio,Macro,🟡 Moderat,6,25.0,4,2,0,1,2,3,+0.100,+0.02,1.7628,52.44
8,CPI,Macro,⚠️ Lemah,6,25.0,3,4,1,3,2,1,-0.019,+0.29,1.7661,52.32
9,USDIDR,Macro,⚠️ Lemah,8,33.3,3,6,1,4,4,0,-0.281,+0.16,1.7737,51.65
10,Nickel,Commodity,⚠️ Lemah,7,29.2,5,2,0,0,4,3,-0.382,-0.62,1.7728,51.83



  REKOMENDASI COVARIATE UNTUK TRAINING

✅ Kuat:
  Silver             [Commodity]  Pass=16/24  MAPEImpr=+0.637%  DA=+0.91pp  MASE=1.7528
  WTI                [Commodity]  Pass=13/24  MAPEImpr=+0.190%  DA=+0.73pp  MASE=1.7618

🟡 Moderat:
  Gold               [Commodity]  Pass=11/24  MAPEImpr=+0.402%  DA=+0.15pp  MASE=1.7575
  STI                [Regional ]  Pass=9/24  MAPEImpr=+0.317%  DA=-0.23pp  MASE=1.7588
  Coal               [Commodity]  Pass=7/24  MAPEImpr=+0.107%  DA=-0.05pp  MASE=1.7638
  Tin                [Commodity]  Pass=6/24  MAPEImpr=+0.105%  DA=-0.17pp  MASE=1.7644
  NPL_Ratio          [Macro    ]  Pass=6/24  MAPEImpr=+0.100%  DA=+0.02pp  MASE=1.7628

⚠️ Lemah:
  CPI                [Macro    ]  Pass=6/24  MAPEImpr=-0.019%  DA=+0.29pp  MASE=1.7661
  USDIDR             [Macro    ]  Pass=8/24  MAPEImpr=-0.281%  DA=+0.16pp  MASE=1.7737
  Nickel             [Commodity]  Pass=7/24  MAPEImpr=-0.382%  DA=-0.62pp  MASE=1.7728

❌ Tidak lolos:
  M2                 [Macro    ]  Pass=

---
## 14. Training Group Covariates (Berdasarkan Screening)

Training model dengan 6 konfigurasi covariate yang dipilih dari hasil screening:

| Group | Isi | Jumlah |
|---|---|---|
| **Baseline** | Tanpa covariate | 0 |
| **Screening1** | Silver, WTI, Gold, STI, Coal, Tin, NPL_Ratio | 7 |
| **Screening2** | Screening1 + CPI, USDIDR, Nickel | 10 |
| **All_Commodity_STI** | Semua komoditas + STI | 8 |
| **All_Macro_no_UST** | Semua makro kecuali US_Treasury_10Y | 6 |
| **All_Covariates** | Semua 15 variabel | 15 |

Total: 6 × 4 model × 2 window × 3 horizon = **144 eksperimen**

In [25]:
# ── Definisi Group Covariates ──────────────────────────────────────────────
GROUP_COVARIATES = {
    "Baseline": [],
    "Screening1": [
        "Silver", "WTI", "Gold", "STI",
        "Coal", "Tin", "NPL_Ratio",
    ],
    "Screening2": [
        "Silver", "WTI", "Gold", "STI",
        "Coal", "Tin", "NPL_Ratio",
        "CPI", "USDIDR", "Nickel",
    ],
    "All_Commodity_STI": [
        "Coal", "Copper", "Nickel", "Silver", "Tin", "Gold", "WTI", "STI",
    ],
    "All_Macro_no_UST": [
        "BI_Rate", "CPI", "M2", "NPL_Ratio", "USDIDR", "GDP",
    ],
    "All_Covariates": [
        "BI_Rate", "CPI", "M2", "NPL_Ratio", "USDIDR", "GDP",
        "Coal", "Copper", "Nickel", "Silver", "Tin", "Gold", "WTI", "STI",
        "US_Treasury_10Y",
    ],
}

total_exp = len(GROUP_COVARIATES) * 4 * len(WINDOWS) * len(HORIZONS)
print("Group Covariates:")
for k, v in GROUP_COVARIATES.items():
    print(f"  {k:<22} ({len(v):>2} vars): {v}")
print(f"\nTotal eksperimen: {len(GROUP_COVARIATES)} groups × 4 models × {len(WINDOWS)} windows × {len(HORIZONS)} horizons = {total_exp}")

Group Covariates:
  Baseline               ( 0 vars): []
  Screening1             ( 7 vars): ['Silver', 'WTI', 'Gold', 'STI', 'Coal', 'Tin', 'NPL_Ratio']
  Screening2             (10 vars): ['Silver', 'WTI', 'Gold', 'STI', 'Coal', 'Tin', 'NPL_Ratio', 'CPI', 'USDIDR', 'Nickel']
  All_Commodity_STI      ( 8 vars): ['Coal', 'Copper', 'Nickel', 'Silver', 'Tin', 'Gold', 'WTI', 'STI']
  All_Macro_no_UST       ( 6 vars): ['BI_Rate', 'CPI', 'M2', 'NPL_Ratio', 'USDIDR', 'GDP']
  All_Covariates         (15 vars): ['BI_Rate', 'CPI', 'M2', 'NPL_Ratio', 'USDIDR', 'GDP', 'Coal', 'Copper', 'Nickel', 'Silver', 'Tin', 'Gold', 'WTI', 'STI', 'US_Treasury_10Y']

Total eksperimen: 6 groups × 4 models × 2 windows × 3 horizons = 144


In [26]:
MODEL_NAMES = ["RandomForest", "ExtraTrees", "XGBoost", "LightGBM"]
group_results = []
group_failed  = []
exp_num       = 0
start_time    = datetime.now()

print(f"Started : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total   : {total_exp} experiments")
print("=" * 82)

for model_name in MODEL_NAMES:
    for grp_name, grp_vars in GROUP_COVARIATES.items():
        for window in WINDOWS:
            for horizon in HORIZONS:
                exp_num += 1
                tag = (f"[{exp_num:>3}/{total_exp}] "
                       f"{model_name:13s} | {grp_name:22s} | W{window:>3}_H{horizon:>2}")
                print(f"{tag}", end=" ... ", flush=True)
                try:
                    m = evaluate_model(model_name, grp_name, grp_vars, window, horizon)
                    group_results.append({
                        "Model": model_name, "Covariates": grp_name,
                        "Window": window, "Horizon": horizon, **m,
                    })
                    print(f"MAPE={m['mape']:.4f}%  DA={m['da']:.1f}%  MASE={m['mase']:.4f}")
                except Exception as e:
                    group_failed.append({"tag": tag, "error": str(e)})
                    print(f"FAILED: {e}")
                    traceback.print_exc()
                finally:
                    gc.collect()

        # Auto-save per model selesai satu group
        if group_results:
            pd.DataFrame(group_results).to_csv("phase1a_group_results.csv", index=False)
            print(f"  → Checkpoint saved ({len(group_results)} rows)")

elapsed = datetime.now() - start_time
print("=" * 82)
print(f"Done in {elapsed} | {len(group_results)} OK, {len(group_failed)} failed")

df_groups = pd.DataFrame(group_results)
df_groups.to_csv("phase1a_group_results.csv", index=False)
print(f"Saved: phase1a_group_results.csv ({len(df_groups)} rows)")

Started : 2026-05-09 15:02:55
Total   : 144 experiments
[  1/144] RandomForest  | Baseline               | W 20_H 1 ... MAPE=0.5072%  DA=51.7%  MASE=0.9746
[  2/144] RandomForest  | Baseline               | W 20_H 5 ... MAPE=0.8522%  DA=47.5%  MASE=1.6371
[  3/144] RandomForest  | Baseline               | W 20_H20 ... MAPE=1.3920%  DA=49.9%  MASE=2.6870
[  4/144] RandomForest  | Baseline               | W120_H 1 ... MAPE=0.5069%  DA=52.3%  MASE=0.9740
[  5/144] RandomForest  | Baseline               | W120_H 5 ... MAPE=0.8526%  DA=47.9%  MASE=1.6378
[  6/144] RandomForest  | Baseline               | W120_H20 ... MAPE=1.3892%  DA=49.3%  MASE=2.6809
  → Checkpoint saved (6 rows)
[  7/144] RandomForest  | Screening1             | W 20_H 1 ... MAPE=0.5047%  DA=53.8%  MASE=0.9699
[  8/144] RandomForest  | Screening1             | W 20_H 5 ... MAPE=0.8519%  DA=49.0%  MASE=1.6367
[  9/144] RandomForest  | Screening1             | W 20_H20 ... MAPE=1.4050%  DA=49.3%  MASE=2.7124
[ 10/144] Rand

In [31]:
# ── Ringkasan hasil ────────────────────────────────────────────────────────
df_groups = pd.read_csv("phase1a_group_results.csv")

# Baseline lookup per Model × Window × Horizon
bl_grp = (
    df_groups[df_groups["Covariates"] == "Baseline"]
    .set_index(["Model", "Window", "Horizon"])[["mape", "da"]]
)

# Improvement vs baseline per baris
rows_cmp = []
for _, r in df_groups.iterrows():
    key = (r["Model"], r["Window"], r["Horizon"])
    if key in bl_grp.index and r["Covariates"] != "Baseline":
        bl_mape = bl_grp.loc[key, "mape"]
        bl_da   = bl_grp.loc[key, "da"]
        impr_mape = (bl_mape - r["mape"]) / bl_mape * 100
        impr_da   = r["da"] - bl_da
    else:
        impr_mape = 0.0
        impr_da   = 0.0
    rows_cmp.append({**r.to_dict(),
                     "impr_mape%": round(impr_mape, 3),
                     "impr_da_pp": round(impr_da, 2)})

df_grp_cmp = pd.DataFrame(rows_cmp)

GROUP_ORDER = ["Baseline","Screening1","Screening2",
               "All_Commodity_STI","All_Macro_no_UST","All_Covariates"]

# ── Tabel 1: Avg MAPE & DA per Group (avg semua model × skenario) ──────────
t1 = (
    df_grp_cmp.groupby("Covariates")
    .agg(avg_mape=("mape","mean"), avg_da=("da","mean"),
         avg_impr_mape=("impr_mape%","mean"), avg_impr_da=("impr_da_pp","mean"))
    .round({"avg_mape":4,"avg_da":2,"avg_impr_mape":3,"avg_impr_da":2})
    .reindex([g for g in GROUP_ORDER if g in df_grp_cmp["Covariates"].unique()])
)

print("="*60)
print("  Tabel 1 — Avg MAPE & DA per Group (avg 4 model × 6 skenario)")
print("="*60)
display(
    t1.rename(columns={
        "avg_mape":"Avg MAPE (%)","avg_da":"Avg DA (%)",
        "avg_impr_mape":"Impr MAPE (%)","avg_impr_da":"Impr DA (pp)",
    })
    .style
    .format({
        "Avg MAPE (%)":"{:.4f}","Avg DA (%)":"{:.2f}",
        "Impr MAPE (%)":"{:+.3f}","Impr DA (pp)":"{:+.2f}",
    })
    .apply(lambda col:[
        "background-color:#e8f5e9" if v>0 else
        "background-color:#ffebee" if v<0 else ""
        for v in col], subset=["Impr MAPE (%)"])
    .apply(lambda col:[
        "background-color:#e3f2fd" if v>0 else
        "background-color:#fff3e0" if v<0 else ""
        for v in col], subset=["Impr DA (pp)"])
    .set_caption("Tabel 1: Avg MAPE & DA per Group Covariate (avg semua model & skenario)")
)

# ── Tabel 2: Pivot Group × Model ───────────────────────────────────────────
t2_raw = (
    df_grp_cmp.groupby(["Covariates","Model"])
    .agg(avg_mape=("mape","mean"), avg_da=("da","mean"))
    .round({"avg_mape":4,"avg_da":2})
    .reset_index()
)

MODEL_ORDER = ["RandomForest","ExtraTrees","XGBoost","LightGBM"]

t2_mape = (t2_raw.pivot(index="Covariates", columns="Model", values="avg_mape")
           .reindex(index=GROUP_ORDER, columns=MODEL_ORDER))
t2_mape.columns = [f"{c}\nMAPE%" for c in t2_mape.columns]

t2_da = (t2_raw.pivot(index="Covariates", columns="Model", values="avg_da")
         .reindex(index=GROUP_ORDER, columns=MODEL_ORDER))
t2_da.columns = [f"{c}\nDA%" for c in t2_da.columns]

t2 = pd.concat([t2_mape, t2_da], axis=1)
print("\n"+"="*60)
print("  Tabel 2 — Avg MAPE & DA per Group × Model")
print("="*60)
display(
    t2.style
    .format({c:"{:.4f}" for c in t2.columns if "MAPE" in c})
    .format({c:"{:.2f}"  for c in t2.columns if "DA"   in c})
    .set_caption("Tabel 2: Avg MAPE & DA per Group × Algoritma")
)

# ── Top 5 overall ───────────────────────────────────────────────────────────
print("\n"+"="*60)
print("  Top 5 Overall (MAPE terkecil, non-baseline)")
print("="*60)

bl_global = df_grp_cmp[df_grp_cmp["Covariates"]=="Baseline"]["mape"].mean()
top5 = (
    df_grp_cmp[df_grp_cmp["Covariates"] != "Baseline"]
    .sort_values("mape").head(5)
    [["Model","Covariates","Window","Horizon","mape","da","impr_mape%","impr_da_pp","mase"]]
    .reset_index(drop=True)
)
top5.index += 1
display(
    top5.rename(columns={
        "mape":"MAPE (%)","da":"DA (%)","impr_mape%":"Impr MAPE %",
        "impr_da_pp":"Impr DA (pp)","mase":"MASE",
    })
    .style.format({
        "MAPE (%)":"{:.4f}","DA (%)":"{:.2f}",
        "Impr MAPE %":"{:+.3f}","Impr DA (pp)":"{:+.2f}","MASE":"{:.4f}",
    })
    .bar(subset=["Impr MAPE %"], color="#a5d6a7", vmin=0)
    .set_caption(f"Top 5 Overall — Baseline global avg = {bl_global:.4f}%")
)

  Tabel 1 — Avg MAPE & DA per Group (avg 4 model × 6 skenario)


,Avg MAPE (%),Avg DA (%),Impr MAPE (%),Impr DA (pp)
Covariates,,,,
Baseline,0.9169,49.63,+0.000,+0.00
Screening1,0.9047,50.16,+1.232,+0.53
Screening2,0.9080,50.06,+0.967,+0.43
All_Commodity_STI,0.9071,49.89,+0.998,+0.26
All_Macro_no_UST,0.9185,49.74,-0.065,+0.11
All_Covariates,0.9201,49.71,-0.005,+0.09



  Tabel 2 — Avg MAPE & DA per Group × Model


,RandomForest MAPE%,ExtraTrees MAPE%,XGBoost MAPE%,LightGBM MAPE%,RandomForest DA%,ExtraTrees DA%,XGBoost DA%,LightGBM DA%
Covariates,,,,,,,,
Baseline,0.916700,0.914400,0.919400,0.917200,49.77,49.99,49.23,49.52
Screening1,0.918300,0.915000,0.893800,0.891800,50.44,49.61,49.80,50.78
Screening2,0.918900,0.914600,0.899600,0.899100,50.38,50.40,50.15,49.29
All_Commodity_STI,0.920100,0.915400,0.899400,0.893600,49.86,50.43,49.51,49.74
All_Macro_no_UST,0.914800,0.914900,0.923000,0.921400,50.47,49.87,49.48,49.13
All_Covariates,0.945000,0.917600,0.913000,0.904900,49.74,49.99,49.38,49.74



  Top 5 Overall (MAPE terkecil, non-baseline)


,Model,Covariates,Window,Horizon,MAPE (%),DA (%),Impr MAPE %,Impr DA (pp),MASE
1,LightGBM,Screening1,120,1,0.5007,55.89,+1.804,+5.32,0.9619
2,XGBoost,Screening1,120,1,0.5012,51.90,+2.642,+3.80,0.9627
3,XGBoost,Screening1,20,1,0.5017,54.37,+0.535,-0.38,0.9636
4,LightGBM,Screening1,20,1,0.5026,54.75,+0.396,+0.57,0.9655
5,LightGBM,All_Macro_no_UST,20,1,0.5026,53.80,+0.396,-0.38,0.9655


In [32]:

# Pastikan df_grp_cmp sudah ada dari cell sebelumnya
# Jika belum, load dulu:
try:
    df_grp_cmp
except NameError:
    import pandas as pd, numpy as np
    df_groups  = pd.read_csv("phase1a_group_results.csv")
    bl_grp = df_groups[df_groups["Covariates"]=="Baseline"].set_index(["Model","Window","Horizon"])[["mape","da"]]
    rows = []
    for _, r in df_groups.iterrows():
        key = (r["Model"], r["Window"], r["Horizon"])
        if key in bl_grp.index and r["Covariates"] != "Baseline":
            impr_mape = (bl_grp.loc[key,"mape"] - r["mape"]) / bl_grp.loc[key,"mape"] * 100
            impr_da   = r["da"] - bl_grp.loc[key,"da"]
        else:
            impr_mape = impr_da = 0.0
        rows.append({**r.to_dict(), "impr_mape%": round(impr_mape,3), "impr_da_pp": round(impr_da,2)})
    df_grp_cmp = pd.DataFrame(rows)

GROUP_ORDER = ["Baseline","Screening1","Screening2",
               "All_Commodity_STI","All_Macro_no_UST","All_Covariates"]
MODEL_ORDER = ["RandomForest","ExtraTrees","XGBoost","LightGBM"]

# ══════════════════════════════════════════════════════════════════════════════
# Tabel 1 — Avg, Best MAPE & DA + N per Model ML
# ══════════════════════════════════════════════════════════════════════════════
t1 = (
    df_grp_cmp
    .groupby("Model")
    .agg(
        avg_mape  = ("mape", "mean"),
        best_mape = ("mape", "min"),
        avg_da    = ("da",   "mean"),
        best_da   = ("da",   "max"),
        n         = ("mape", "count"),
    )
    .round({"avg_mape":4,"best_mape":4,"avg_da":2,"best_da":2})
    .reindex([m for m in MODEL_ORDER if m in df_grp_cmp["Model"].unique()])
)

print("="*60)
print("  Tabel 1 — Avg & Best MAPE/DA per Model")
print("  (semua group × semua skenario)")
print("="*60)
display(
    t1.rename(columns={
        "avg_mape":"Avg MAPE (%)","best_mape":"Best MAPE (%)",
        "avg_da":"Avg DA (%)","best_da":"Best DA (%)","n":"N",
    })
    .style
    .format({
        "Avg MAPE (%)":"{:.4f}","Best MAPE (%)":"{:.4f}",
        "Avg DA (%)":"{:.2f}", "Best DA (%)":"{:.2f}",
    })
    .highlight_min(subset=["Avg MAPE (%)","Best MAPE (%)"], color="#c8e6c9")
    .highlight_max(subset=["Best DA (%)","Avg DA (%)"],     color="#bbdefb")
    .set_caption("Tabel 1: Avg & Best MAPE/DA per Algoritma — Group Experiments")
)

# ══════════════════════════════════════════════════════════════════════════════
# Tabel 2 — Avg MAPE & DA per Skenario × Model (pivot)
# ══════════════════════════════════════════════════════════════════════════════
df_grp_cmp["Skenario"] = "W" + df_grp_cmp["Window"].astype(str) + "_H" + df_grp_cmp["Horizon"].astype(str)
SCENARIO_ORDER = ["W20_H1","W120_H1","W20_H5","W120_H5","W20_H20","W120_H20"]

t2_raw = (
    df_grp_cmp
    .groupby(["Skenario","Model"])
    .agg(avg_mape=("mape","mean"), avg_da=("da","mean"))
    .round({"avg_mape":4,"avg_da":2})
    .reset_index()
)

t2_mape = (t2_raw.pivot(index="Skenario", columns="Model", values="avg_mape")
           .reindex(index=SCENARIO_ORDER, columns=MODEL_ORDER))
t2_mape.columns = [f"{c}\nMAPE%" for c in t2_mape.columns]

t2_da = (t2_raw.pivot(index="Skenario", columns="Model", values="avg_da")
         .reindex(index=SCENARIO_ORDER, columns=MODEL_ORDER))
t2_da.columns = [f"{c}\nDA%" for c in t2_da.columns]

t2 = pd.concat([t2_mape, t2_da], axis=1)

print("\n"+"="*60)
print("  Tabel 2 — Avg MAPE & DA per Skenario × Model")
print("  (avg semua group pada skenario tersebut)")
print("="*60)
display(
    t2.style
    .format({c:"{:.4f}" for c in t2.columns if "MAPE" in c})
    .format({c:"{:.2f}"  for c in t2.columns if "DA"   in c})
    .set_caption("Tabel 2: Avg MAPE & DA per Skenario × Algoritma — Group Experiments")
)


  Tabel 1 — Avg & Best MAPE/DA per Model
  (semua group × semua skenario)


,Avg MAPE (%),Best MAPE (%),Avg DA (%),Best DA (%),N
Model,,,,,
RandomForest,0.9223,0.5047,50.11,53.80,36
ExtraTrees,0.9153,0.5047,50.05,54.75,36
XGBoost,0.9080,0.5012,49.59,54.94,36
LightGBM,0.9047,0.5007,49.70,55.89,36



  Tabel 2 — Avg MAPE & DA per Skenario × Model
  (avg semua group pada skenario tersebut)


,RandomForest MAPE%,ExtraTrees MAPE%,XGBoost MAPE%,LightGBM MAPE%,RandomForest DA%,ExtraTrees DA%,XGBoost DA%,LightGBM DA%
Skenario,,,,,,,,
W20_H1,0.506000,0.505200,0.504200,0.504100,52.15,53.23,54.28,53.74
W120_H1,0.506500,0.505200,0.507800,0.505900,52.63,52.98,50.41,51.90
W20_H5,0.855000,0.853000,0.828200,0.833000,48.50,47.96,48.25,47.49
W120_H5,0.855400,0.852400,0.842100,0.837200,48.28,48.12,48.18,48.19
W20_H20,1.411300,1.391500,1.389000,1.388000,49.58,48.62,48.49,49.04
W120_H20,1.399700,1.384500,1.376800,1.360000,49.52,49.39,47.94,47.85


In [33]:

def tabel_full_per_model(model_name):
    """
    Tabel lengkap: setiap baris = Group Covariate, setiap kolom = skenario W×H.
    Baseline selalu di baris pertama. Nilai = MAPE dan DA.
    Juga tampilkan improvement vs baseline per skenario.
    """
    TAG = {"RandomForest":"RF","ExtraTrees":"ET","XGBoost":"XGB","LightGBM":"LGB"}
    tag = TAG[model_name]

    sub = df_grp_cmp[df_grp_cmp["Model"] == model_name].copy()
    sub["Skenario"] = "W" + sub["Window"].astype(str) + "_H" + sub["Horizon"].astype(str)

    # ── MAPE pivot ──────────────────────────────────────────────────────────
    mape_piv = (
        sub.pivot_table(index="Covariates", columns="Skenario", values="mape")
        .reindex(index=[g for g in GROUP_ORDER if g in sub["Covariates"].unique()],
                 columns=SCENARIO_ORDER)
        .round(4)
    )

    # ── DA pivot ────────────────────────────────────────────────────────────
    da_piv = (
        sub.pivot_table(index="Covariates", columns="Skenario", values="da")
        .reindex(index=[g for g in GROUP_ORDER if g in sub["Covariates"].unique()],
                 columns=SCENARIO_ORDER)
        .round(2)
    )

    # ── Impr MAPE pivot ─────────────────────────────────────────────────────
    impr_piv = (
        sub.pivot_table(index="Covariates", columns="Skenario", values="impr_mape%")
        .reindex(index=[g for g in GROUP_ORDER if g in sub["Covariates"].unique()],
                 columns=SCENARIO_ORDER)
        .round(3)
    )

    # ── Impr DA pivot ───────────────────────────────────────────────────────
    impr_da_piv = (
        sub.pivot_table(index="Covariates", columns="Skenario", values="impr_da_pp")
        .reindex(index=[g for g in GROUP_ORDER if g in sub["Covariates"].unique()],
                 columns=SCENARIO_ORDER)
        .round(2)
    )

    print(f"\n{'#'*70}")
    print(f"  [{tag}] {model_name}")
    print(f"{'#'*70}")

    # Sub-tabel A: MAPE per group × skenario
    print(f"\n  [A] MAPE (%) — {model_name}")
    mape_piv.columns.name = None
    mape_piv.index.name   = "Group"
    display(
        mape_piv.style
        .format("{:.4f}")
        .apply(lambda col: [
            "background-color:#f5f5f5;font-weight:bold" if idx=="Baseline" else ""
            for idx in mape_piv.index], axis=0)
        .highlight_min(axis=0, color="#c8e6c9")
        .set_caption(f"[{tag}] MAPE (%) per Group × Skenario")
    )

    # Sub-tabel B: DA per group × skenario
    print(f"\n  [B] DA (%) — {model_name}")
    da_piv.columns.name = None
    da_piv.index.name   = "Group"
    display(
        da_piv.style
        .format("{:.2f}")
        .apply(lambda col: [
            "background-color:#f5f5f5;font-weight:bold" if idx=="Baseline" else ""
            for idx in da_piv.index], axis=0)
        .highlight_max(axis=0, color="#bbdefb")
        .set_caption(f"[{tag}] DA (%) per Group × Skenario")
    )

    # Sub-tabel C: Improvement MAPE
    print(f"\n  [C] Improvement MAPE (%) vs Baseline — {model_name}")
    impr_piv.columns.name = None
    impr_piv.index.name   = "Group"
    def color_impr_mape(df):
        styles = pd.DataFrame("", index=df.index, columns=df.columns)
        for col in df.columns:
            for idx in df.index:
                v = df.loc[idx, col]
                if pd.isna(v) or idx == "Baseline":
                    styles.loc[idx, col] = "background-color:#f5f5f5"
                elif v > 0:
                    styles.loc[idx, col] = "background-color:#e8f5e9;color:#2e7d32;font-weight:bold"
                elif v < 0:
                    styles.loc[idx, col] = "background-color:#ffebee;color:#c62828"
        return styles
    display(
        impr_piv.style
        .format("{:+.3f}")
        .apply(color_impr_mape, axis=None)
        .set_caption(f"[{tag}] Improvement MAPE (%) vs Baseline — hijau=lebih baik, merah=lebih buruk")
    )

    # Sub-tabel D: Improvement DA
    print(f"\n  [D] Improvement DA (pp) vs Baseline — {model_name}")
    impr_da_piv.columns.name = None
    impr_da_piv.index.name   = "Group"
    def color_impr_da(df):
        styles = pd.DataFrame("", index=df.index, columns=df.columns)
        for col in df.columns:
            for idx in df.index:
                v = df.loc[idx, col]
                if pd.isna(v) or idx == "Baseline":
                    styles.loc[idx, col] = "background-color:#f5f5f5"
                elif v > 0:
                    styles.loc[idx, col] = "background-color:#e3f2fd;color:#1565c0;font-weight:bold"
                elif v < 0:
                    styles.loc[idx, col] = "background-color:#fff3e0;color:#e65100"
        return styles
    display(
        impr_da_piv.style
        .format("{:+.2f}")
        .apply(color_impr_da, axis=None)
        .set_caption(f"[{tag}] Improvement DA (pp) vs Baseline — biru=lebih baik, oranye=lebih buruk")
    )


for model_name in MODEL_ORDER:
    if model_name in df_grp_cmp["Model"].unique():
        tabel_full_per_model(model_name)



######################################################################
  [RF] RandomForest
######################################################################

  [A] MAPE (%) — RandomForest


,W20_H1,W120_H1,W20_H5,W120_H5,W20_H20,W120_H20
Group,,,,,,
Baseline,0.5072,0.5069,0.8522,0.8526,1.3920,1.3892
Screening1,0.5047,0.5059,0.8519,0.8526,1.4050,1.3900
Screening2,0.5053,0.5059,0.8528,0.8546,1.4037,1.3909
All_Commodity_STI,0.5057,0.5060,0.8521,0.8526,1.4051,1.3991
All_Macro_no_UST,0.5062,0.5069,0.8553,0.8556,1.3870,1.3779
All_Covariates,0.5067,0.5072,0.8656,0.8643,1.4750,1.4511



  [B] DA (%) — RandomForest


,W20_H1,W120_H1,W20_H5,W120_H5,W20_H20,W120_H20
Group,,,,,,
Baseline,51.71,52.28,47.52,47.90,49.90,49.33
Screening1,53.80,53.23,49.05,47.71,49.33,49.52
Screening2,50.95,53.04,49.43,49.05,49.71,50.10
All_Commodity_STI,52.47,51.90,48.47,48.09,49.13,49.13
All_Macro_no_UST,52.85,53.04,48.85,48.28,49.71,50.10
All_Covariates,51.14,52.28,47.71,48.66,49.71,48.94



  [C] Improvement MAPE (%) vs Baseline — RandomForest


...

,W20_H1,W120_H1,W20_H5,W120_H5,W20_H20,W120_H20
Group,,,,,,
Baseline,54.18,50.57,48.28,47.52,49.33,47.21
Screening1,54.75,55.89,47.52,48.09,49.71,48.75
Screening2,53.61,51.33,47.14,47.90,48.36,47.40
All_Commodity_STI,53.23,52.47,46.76,48.28,49.52,48.17
All_Macro_no_UST,53.80,47.91,48.28,48.09,49.13,47.59
All_Covariates,52.85,53.23,46.95,49.24,48.17,47.98



  [C] Improvement MAPE (%) vs Baseline — LightGBM


,W20_H1,W120_H1,W20_H5,W120_H5,W20_H20,W120_H20
Group,,,,,,
Baseline,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000
Screening1,+0.396,+1.804,+3.001,+2.044,+2.765,+4.296
Screening2,+0.218,+0.490,+4.056,+2.972,+1.122,+2.130
All_Commodity_STI,-0.258,+1.098,+3.013,+2.878,+2.479,+3.798
All_Macro_no_UST,+0.396,+0.000,+0.856,-0.352,-0.765,-1.459
All_Covariates,-0.178,+1.294,+3.224,+2.490,-0.614,+2.036



  [D] Improvement DA (pp) vs Baseline — LightGBM


,W20_H1,W120_H1,W20_H5,W120_H5,W20_H20,W120_H20
Group,,,,,,
Baseline,+0.00,+0.00,+0.00,+0.00,+0.00,+0.00
Screening1,+0.57,+5.32,-0.76,+0.57,+0.38,+1.54
Screening2,-0.57,+0.76,-1.14,+0.38,-0.97,+0.19
All_Commodity_STI,-0.95,+1.90,-1.52,+0.76,+0.19,+0.96
All_Macro_no_UST,-0.38,-2.66,+0.00,+0.57,-0.20,+0.38
All_Covariates,-1.33,+2.66,-1.33,+1.72,-1.16,+0.77


In [34]:

SCENARIO_ORDER = ["W20_H1","W120_H1","W20_H5","W120_H5","W20_H20","W120_H20"]
MODEL_ORDER    = ["RandomForest","ExtraTrees","XGBoost","LightGBM"]
MODEL_TAG      = {"RandomForest":"RF","ExtraTrees":"ET","XGBoost":"XGB","LightGBM":"LGB"}
GROUP_ORDER    = ["Baseline","Screening1","Screening2",
                  "All_Commodity_STI","All_Macro_no_UST","All_Covariates"]

df_grp_cmp["Skenario"] = ("W" + df_grp_cmp["Window"].astype(str)
                          + "_H" + df_grp_cmp["Horizon"].astype(str))

def color_type(val):
    return {
        "Best":     "background-color:#c8e6c9;font-weight:bold;color:#1b5e20",
        "Baseline": "background-color:#f5f5f5;font-style:italic",
        "Worst":    "background-color:#ffcdd2;color:#b71c1c",
    }.get(val, "")

def color_impr(val):
    try:
        v = float(val)
        if   v > 0: return "color:#2e7d32;font-weight:bold"
        elif v < 0: return "color:#c62828"
        return ""
    except: return ""

for sc in SCENARIO_ORDER:
    rows = []
    sub_sc = df_grp_cmp[df_grp_cmp["Skenario"] == sc]

    for model_name in MODEL_ORDER:
        sub_m = sub_sc[sub_sc["Model"] == model_name]
        if sub_m.empty:
            continue

        # Baseline
        bl_row = sub_m[sub_m["Covariates"] == "Baseline"]
        if bl_row.empty:
            continue
        bl = bl_row.iloc[0]

        # Non-baseline
        non_bl = sub_m[sub_m["Covariates"] != "Baseline"]
        if non_bl.empty:
            continue

        best_row  = non_bl.loc[non_bl["mape"].idxmin()]
        worst_row = non_bl.loc[non_bl["mape"].idxmax()]

        for label, row in [("Best", best_row), ("Baseline", bl), ("Worst", worst_row)]:
            rows.append({
                "Model":       MODEL_TAG[model_name],
                "Type":        label,
                "Group":       row["Covariates"],
                "MAPE (%)":    round(row["mape"], 4),
                "DA (%)":      round(row["da"],   2),
                "Impr MAPE%":  round(row["impr_mape%"], 3) if label != "Baseline" else "-",
                "Impr DA(pp)": round(row["impr_da_pp"], 2) if label != "Baseline" else "-",
            })

    tbl = pd.DataFrame(rows)

    w, h = int(sc.split("_H")[0][1:]), int(sc.split("_H")[1])
    print(f"\n{'='*70}")
    print(f"  Skenario W{w}_H{h}")
    print(f"  Best / Baseline / Worst per Algoritma (MAPE terkecil = Best)")
    print(f"{'='*70}")

    display(
        tbl.reset_index(drop=True)
        .style
        .map(color_type, subset=["Type"])
        .map(color_impr, subset=["Impr MAPE%","Impr DA(pp)"])
        .format({
            "MAPE (%)": "{:.4f}",
            "DA (%)":   "{:.2f}",
        })
        .format(lambda v: f"{v:+.3f}" if isinstance(v, float) else v,
                subset=["Impr MAPE%"])
        .format(lambda v: f"{v:+.2f}" if isinstance(v, float) else v,
                subset=["Impr DA(pp)"])
        .set_caption(
            f"W{w}_H{h} — Best (hijau) / Baseline (abu) / Worst (merah) "
            f"per algoritma. Impr vs Baseline masing-masing model."
        )
    )



  Skenario W20_H1
  Best / Baseline / Worst per Algoritma (MAPE terkecil = Best)


,Model,Type,Group,MAPE (%),DA (%),Impr MAPE%,Impr DA(pp)
0,RF,Best,Screening1,0.5047,53.80,+0.493,+2.09
1,RF,Baseline,Baseline,0.5072,51.71,-,-
2,RF,Worst,All_Covariates,0.5067,51.14,+0.099,-0.57
3,ET,Best,All_Commodity_STI,0.5047,53.80,+0.119,+0.19
4,ET,Baseline,Baseline,0.5053,53.61,-,-
5,ET,Worst,All_Covariates,0.5057,53.61,-0.079,+0.00
6,XGB,Best,Screening1,0.5017,54.37,+0.535,-0.38
7,XGB,Baseline,Baseline,0.5044,54.75,-,-
8,XGB,Worst,All_Covariates,0.5069,53.61,-0.496,-1.14
9,LGB,Best,Screening1,0.5026,54.75,+0.396,+0.57



  Skenario W120_H1
  Best / Baseline / Worst per Algoritma (MAPE terkecil = Best)


,Model,Type,Group,MAPE (%),DA (%),Impr MAPE%,Impr DA(pp)
0,RF,Best,Screening1,0.5059,53.23,+0.197,+0.95
1,RF,Baseline,Baseline,0.5069,52.28,-,-
2,RF,Worst,All_Covariates,0.5072,52.28,-0.059,+0.00
3,ET,Best,Screening2,0.5047,54.75,+0.217,+2.47
4,ET,Baseline,Baseline,0.5058,52.28,-,-
5,ET,Worst,All_Macro_no_UST,0.5055,51.71,+0.059,-0.57
6,XGB,Best,Screening1,0.5012,51.90,+2.642,+3.80
7,XGB,Baseline,Baseline,0.5148,48.10,-,-
8,XGB,Worst,All_Macro_no_UST,0.5144,48.86,+0.078,+0.76
9,LGB,Best,Screening1,0.5007,55.89,+1.804,+5.32



  Skenario W20_H5
  Best / Baseline / Worst per Algoritma (MAPE terkecil = Best)


...

,Model,Type,Group,MAPE (%),DA (%),Impr MAPE%,Impr DA(pp)
0,RF,Best,Screening1,0.8526,47.71,+0.000,-0.19
1,RF,Baseline,Baseline,0.8526,47.90,-,-
2,RF,Worst,All_Covariates,0.8643,48.66,-1.372,+0.76
3,ET,Best,Screening1,0.8500,47.71,+0.328,-0.57
4,ET,Baseline,Baseline,0.8528,48.28,-,-
5,ET,Worst,All_Covariates,0.8550,48.09,-0.258,-0.19
6,XGB,Best,All_Covariates,0.8295,48.47,+3.659,-0.38
7,XGB,Baseline,Baseline,0.8610,48.85,-,-
8,XGB,Worst,All_Macro_no_UST,0.8612,48.47,-0.023,-0.38
9,LGB,Best,Screening2,0.8261,47.90,+2.972,+0.38



  Skenario W20_H20
  Best / Baseline / Worst per Algoritma (MAPE terkecil = Best)


,Model,Type,Group,MAPE (%),DA (%),Impr MAPE%,Impr DA(pp)
0,RF,Best,All_Macro_no_UST,1.3870,49.71,+0.359,-0.19
1,RF,Baseline,Baseline,1.3920,49.90,-,-
2,RF,Worst,All_Covariates,1.4750,49.71,-5.963,-0.19
3,ET,Best,Screening2,1.3865,48.94,-0.108,+0.19
4,ET,Baseline,Baseline,1.3850,48.75,-,-
5,ET,Worst,All_Covariates,1.4007,48.55,-1.134,-0.20
6,XGB,Best,All_Commodity_STI,1.3739,48.55,+1.407,+0.00
7,XGB,Baseline,Baseline,1.3935,48.55,-,-
8,XGB,Worst,All_Covariates,1.4085,48.94,-1.076,+0.39
9,LGB,Best,Screening1,1.3609,49.71,+2.765,+0.38



  Skenario W120_H20
  Best / Baseline / Worst per Algoritma (MAPE terkecil = Best)


,Model,Type,Group,MAPE (%),DA (%),Impr MAPE%,Impr DA(pp)
0,RF,Best,All_Macro_no_UST,1.3779,50.10,+0.813,+0.77
1,RF,Baseline,Baseline,1.3892,49.33,-,-
2,RF,Worst,All_Covariates,1.4511,48.94,-4.456,-0.39
3,ET,Best,All_Macro_no_UST,1.3818,49.90,+0.181,+0.57
4,ET,Baseline,Baseline,1.3843,49.33,-,-
5,ET,Worst,Screening2,1.3863,49.52,-0.144,+0.19
6,XGB,Best,Screening1,1.3238,48.36,+4.756,+0.58
7,XGB,Baseline,Baseline,1.3899,47.78,-,-
8,XGB,Worst,All_Macro_no_UST,1.4282,47.78,-2.756,+0.00
9,LGB,Best,Screening1,1.3254,48.75,+4.296,+1.54


In [36]:
import pandas as pd
import numpy as np
import joblib
import os

# Load hasil group experiments (Section 14)
df_grp = pd.read_csv("phase1a_group_results.csv")

# Baseline lookup per Model × Window × Horizon
bl_grp = (
    df_grp[df_grp["Covariates"] == "Baseline"]
    .set_index(["Model", "Window", "Horizon"])[["mape", "da"]]
)

# Top 3 non-baseline, sorted MAPE terkecil
top3_grp = (
    df_grp[df_grp["Covariates"] != "Baseline"]
    .sort_values("mape")
    .head(3)[["Model","Covariates","Window","Horizon","mape","da","mase","hit_large"]]
    .reset_index(drop=True)
)
top3_grp.index += 1

top3_grp["baseline_mape"] = top3_grp.apply(
    lambda r: bl_grp.loc[(r["Model"], r["Window"], r["Horizon"]), "mape"]
    if (r["Model"], r["Window"], r["Horizon"]) in bl_grp.index else float("nan"), axis=1
)
top3_grp["baseline_da"] = top3_grp.apply(
    lambda r: bl_grp.loc[(r["Model"], r["Window"], r["Horizon"]), "da"]
    if (r["Model"], r["Window"], r["Horizon"]) in bl_grp.index else float("nan"), axis=1
)
top3_grp["impr_mape%"] = ((top3_grp["baseline_mape"] - top3_grp["mape"]) / top3_grp["baseline_mape"] * 100).round(3)
top3_grp["impr_da_pp"] = (top3_grp["da"] - top3_grp["baseline_da"]).round(2)

print(f"{'='*65}")
print(f"  TOP 3 OVERALL — Group Covariate Experiments")
print(f"  (dari phase1a_group_results.csv)")
print(f"{'='*65}")
display(
    top3_grp[["Model","Covariates","Window","Horizon",
              "baseline_mape","mape","impr_mape%",
              "baseline_da","da","impr_da_pp","mase"]]
    .rename(columns={
        "baseline_mape":"Baseline MAPE (%)","mape":"MAPE (%)",
        "impr_mape%":"Impr MAPE (%)","baseline_da":"Baseline DA (%)",
        "da":"DA (%)","impr_da_pp":"Impr DA (pp)","mase":"MASE",
    })
    .style
    .format({
        "Baseline MAPE (%)":"{:.4f}","MAPE (%)":"{:.4f}",
        "Impr MAPE (%)":"{:+.3f}","Baseline DA (%)":"{:.1f}",
        "DA (%)":"{:.1f}","Impr DA (pp)":"{:+.2f}","MASE":"{:.4f}",
    })
    .bar(subset=["Impr MAPE (%)"], color="#a5d6a7", vmin=0)
    .highlight_min(subset=["MAPE (%)"], color="#c8e6c9")
    .set_caption("Top 3 Group Covariate — akan divisualisasikan di notebook 04_visualization.ipynb")
)

# Simpan untuk notebook 04
GROUP_COV_MAP = {
    "Baseline":        [],
    "Screening1":      ["Silver","WTI","Gold","STI","Coal","Tin","NPL_Ratio"],
    "Screening2":      ["Silver","WTI","Gold","STI","Coal","Tin","NPL_Ratio","CPI","USDIDR","Nickel"],
    "All_Commodity_STI":["Coal","Copper","Nickel","Silver","Tin","Gold","WTI","STI"],
    "All_Macro_no_UST":["BI_Rate","CPI","M2","NPL_Ratio","USDIDR","GDP"],
    "All_Covariates":  ["BI_Rate","CPI","M2","NPL_Ratio","USDIDR","GDP",
                        "Coal","Copper","Nickel","Silver","Tin","Gold","WTI","STI","US_Treasury_10Y"],
}

os.makedirs("saved_models", exist_ok=True)
top3_config = []
for _, row in top3_grp.iterrows():
    top3_config.append({
        "Model":     row["Model"],
        "Covariates": row["Covariates"],
        "Window":    int(row["Window"]),
        "Horizon":   int(row["Horizon"]),
        "cov_vars":  GROUP_COV_MAP.get(row["Covariates"], []),
    })
joblib.dump(top3_config, "saved_models/top3_group_config.joblib")
print(f"\nSaved: saved_models/top3_group_config.joblib")
for i, c in enumerate(top3_config, 1):
    print(f"  #{i}: {c['Model']:14s} | {c['Covariates']:22s} | W{c['Window']}_H{c['Horizon']} | vars={c['cov_vars']}")

  TOP 3 OVERALL — Group Covariate Experiments
  (dari phase1a_group_results.csv)


,Model,Covariates,Window,Horizon,Baseline MAPE (%),MAPE (%),Impr MAPE (%),Baseline DA (%),DA (%),Impr DA (pp),MASE
1,LightGBM,Screening1,120,1,0.5099,0.5007,+1.804,50.6,55.9,+5.32,0.9619
2,XGBoost,Screening1,120,1,0.5148,0.5012,+2.642,48.1,51.9,+3.80,0.9627
3,XGBoost,Screening1,20,1,0.5044,0.5017,+0.535,54.8,54.4,-0.38,0.9636



Saved: saved_models/top3_group_config.joblib
  #1: LightGBM       | Screening1             | W120_H1 | vars=['Silver', 'WTI', 'Gold', 'STI', 'Coal', 'Tin', 'NPL_Ratio']
  #2: XGBoost        | Screening1             | W120_H1 | vars=['Silver', 'WTI', 'Gold', 'STI', 'Coal', 'Tin', 'NPL_Ratio']
  #3: XGBoost        | Screening1             | W20_H1 | vars=['Silver', 'WTI', 'Gold', 'STI', 'Coal', 'Tin', 'NPL_Ratio']
